## Portable path configuration
This release copy contains no personal or NAS paths. Set `GAO2026_DATA_ROOT` for deposited data and `GAO2026_RAW_ROOT` only when running raw-image stages (raw revision microscopy is not included).


In [ ]:
# CSV boundaries accept legacy inputs and export publication state labels.
from pathlib import Path as _StatePath
import sys as _state_sys
_state_start = _StatePath.cwd().resolve()
_state_repo = next((p for p in (_state_start, *_state_start.parents)
                    if (p / "gao2026_state_labels.py").is_file()), None)
if _state_repo is None:
    raise RuntimeError("Run this notebook from within the Gao2026 code repository")
if str(_state_repo) not in _state_sys.path:
    _state_sys.path.insert(0, str(_state_repo))
from gao2026_state_labels import export_transcription_states, read_legacy_state_csv

from pathlib import Path
import os
_default_repo_root = Path.cwd().resolve()
GAO2026_DATA_ROOT = Path(os.environ.get('GAO2026_DATA_ROOT', _default_repo_root / 'data')).expanduser().resolve()
GAO2026_RAW_ROOT = Path(os.environ.get('GAO2026_RAW_ROOT', GAO2026_DATA_ROOT / 'external_raw')).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(os.environ.get('GAO2026_OUTPUT_ROOT', _default_repo_root / 'outputs')).expanduser().resolve()
GAO2026_EXTERNAL_ROOT = Path(os.environ.get('GAO2026_EXTERNAL_ROOT', GAO2026_DATA_ROOT / 'external')).expanduser().resolve()


Note:
- This notebook is part of the Gao et al. (2026, in preparation) analysis workspace:
  Gao et al., Minute-scale coupling of chromatin marks and transcriptional bursts (2026, in preparation)
- For data layout, execution order, and key parameters, see the README.md in this folder:
  - analyses/02_Cross_correlation/README.md
- State labels:
  - Active = ON
  - Inactive = OFF
  CSV exports use Active/Inactive. Internal ON/OFF aliases and legacy filenames are retained only for compatibility.
- Output isolation:
  - All generated files are written below `threshold_sensitivity_outputs_20260811/`
    beside this notebook. The NAS input folders are read-only inputs.
- Center-determination configuration (reviewer R3-m11):
  - PAPER main-text results use the rescue criterion  (USE_TETR_RESCUE = True,
    output folder `.../tetr_mcp_hybrid_analysis`).
  - The `_standard_only` folder is the standard-threshold SENSITIVITY check only.
  - The first code cell now runs BOTH automatically (standard then rescue).


# HDAC inhibitor time-lapse analysis (MCP / mTetR)

This notebook reproduces the analysis pipeline for HDAC inhibitor time-lapse movies described in the manuscript section:
"Analysis of HDAC inhibitor time-lapse data".

Input:
- Two-channel TIFF movies (time × channel × y × x)
  - channel 0: MCP (nascent transcription reporter)
  - channel 1: mTetR (locus marker)
- Optional per-frame cell label images (time × y × x), stored as: <movie_stem>_labels.tif

Output:
- Per-frame Active/Inactive (Active/Inactive) calls per cell
- Per-cell burst metrics (Ton, Toff, burst frequency) estimated with censoring-aware MLE
- Run-level tables (Active/Inactive durations with censoring flags)
- Figure-ready plots (state raster, duty cycle, Kaplan–Meier curves)

Note:
Raw movies are not included in this repository. You must provide local data paths.


## 1. Input data layout

Set ROOT_DIR to a folder that contains time-lapse movies (*.tif). Label masks are optional.

Movies:
- Location: ROOT_DIR/*.tif  (excluding *_labels.tif)
- Expected shape: (T, 2, Y, X)

Optional label masks (per frame):
- File name: <movie_stem>_labels.tif
- Expected shape: (T, Y, X)
- Each pixel stores a cell ID (0 = background)

Condition labels:
Movies are assigned to conditions using the filename prefix:
- "1-" -> DMSO
- "2-" -> TSA
- "3-" -> RGFP966


## 2. Key analysis parameters (report these for reproducibility)

Imaging:
- Frame interval (DELTA_MIN): 1 min
- Pixel size (PIXEL_SIZE_XY_NM): 130 nm

Spot detection (normalized images):
- Detector: Big-FISH detect_spots (recommended / required for reproducibility)
- Main-analysis normalized thresholds: MCP = 1.25; mTetR = 1.10
- R3-m11 sensitivity analysis: one-factor mTetR and MCP threshold sweeps
- Spot radius (SPOT_RADIUS_XY_NM): 200 nm

Tracking / linking:
- Max displacement between consecutive frames (MAX_DISPLACEMENT_PX): 3 px (≈ 390 nm)
- Minimum consecutive frames required (MIN_TRACK_CONSEC): 3

Important:
If Big-FISH is not installed, this notebook can fall back to a simplified detector.
Results may differ substantially from the paper.


## 3. State calling (Active / Inactive) and censoring

Per frame, a cell is classified as Active if:
- An MCP spot is detected within MAX_DISPLACEMENT_PX of the locus position.

Locus position:
- Defined by the mTetR trajectory (one representative mTetR spot per cell per frame).
- If mTetR is missing in a frame, the locus position is inferred from neighboring frames
  using nearest-neighbor linking, and the same proximity criterion is applied.

Noise suppression:
- Single-frame Active calls without temporal support from adjacent frames are relabeled as Inactive.

Run extraction and censoring:
- From the binary time series, contiguous Active and Inactive runs are extracted.
- Runs touching the first frame are marked as left-censored.
- Runs touching the last frame are marked as right-censored.
- Dwell-time distributions are analyzed using Kaplan–Meier curves
  (exclude left-censored runs; treat right-censored runs as censored observations).


## 4. Outputs (saved files)

All generated files are saved beside this notebook under:

- `threshold_sensitivity_outputs_20260811/`

The directory contains the main-analysis outputs, `figs/`, `tables/`, and the
independent mTetR/MCP threshold sweeps under `threshold_sweeps/runs/`. Raw TIFF
movies remain on the NAS and are never modified by this notebook.


## 5. How to run this notebook

1) Run the analysis cell for each gene / locus (e.g., Nanog, Sox2)
   - Set ROOT_DIR and parameters
   - This generates aggregated_perframe.csv and aggregated_runs.csv in outputs

2) Run the "runs_by_cell" cell
   - This creates aggregated_runs_by_cell.csv (recommended for survival analysis)

3) Run the plotting section
   - Generates PDF panels (raster, duty cycle, Kaplan–Meier, QC tables)
   - Outputs are written under figures/ (or a configured FIGDIR)


In [ ]:
# This experiment investigates transcriptional dynamics in mESCs under
# various Epigenetic/HDAC inhibitor conditions:
# 1. DMSO (Control): Vehicle control to establish baseline transcriptional activity.
# 2. TSA: Trichostatin A treatment (Broad-spectrum HDAC inhibitor).
# 3. RGFP966: HDAC3-specific inhibitor treatment.


# Cell 1: Data Processing and Analysis Logic
# 1. Image Pre-processing: Loads 2-channel TIFFs (MCP/TETR) and associated cell labels.
# 2. Normalization & Detection: Uses robust_bg_norm and nucleus_norm_image for
#    background correction, then detect_spots_norm (BigFISH) for spot identification.
# 3. Tracking & Linking: Implements select_single_spot_per_cell to link spots across
#    frames within individual cell masks using distance-based thresholds.
# 4. Quantification: Measures transcriptional burst intensity via 2D Gaussian fitting
#    (measure_spot_intensity_gaussian) on normalized image patches.
# 5. Kinetics Estimation: Calculates On/Off rates and burst frequency using Maximum
#    Likelihood Estimation (mle_rate_discrete) while accounting for censored data.
# 6. Aggregation: Consolidates per-frame, kinetic, and intensity data across all cells
#    and experimental groups into final CSV reports.
#
# ============================================================================
# [FIX SUMMARY] Changes in this cell vs the original (addresses reviewer R3-M4: KM > 60 min):
#   (A) Exclude the self-generated aggregated_*.csv when aggregating, so the glob does not
#       include its own output and double-count rows on a re-run.
#   (B) Drop duplicates on (file, cell_id, frame) after aggregation and check frame uniqueness
#       (warns if duplicates were found).
#   (C) Sort by frame before run extraction, and assert dwell time <= movie duration.
#   (D) Use one shared run extractor for both aggregated_runs.csv and runs_by_cell.
#   Root cause of the old > 60 min KM: duplicated rows became adjacent after sorting,
#   which doubled every dwell time.
# ============================================================================

import os
from pathlib import Path
from glob import glob
import warnings
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import tifffile
from scipy.optimize import minimize_scalar
from scipy.optimize import curve_fit
from scipy import stats as scistats
from skimage.draw import circle_perimeter

warnings.filterwarnings("ignore")


# Configurable Parameters

# ---- Isolated output root beside this notebook ----
# Launch Jupyter from the directory containing this notebook. This guard prevents
# accidental writes to the NAS or to an unrelated working directory.
NOTEBOOK_FILENAME = "inhibitor_response_analysis_revised.ipynb"
NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / NOTEBOOK_FILENAME).exists():
    raise RuntimeError(
        "Launch Jupyter from the notebook directory, or change NOTEBOOK_DIR "
        "explicitly before running this cell."
    )

BASE_DIR = NOTEBOOK_DIR
OUTPUTS_DIR = NOTEBOOK_DIR / "threshold_sensitivity_outputs_20260811"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[output root] {OUTPUTS_DIR}")

DATASETS = [
    ("Nanog", str(GAO2026_RAW_ROOT / 'Public-TS-873-2/Microscope/Gao/250909-live-inhibitor/3.seg_cells')),
    ("Sox2", str(GAO2026_RAW_ROOT / 'Public-TS-873-2/Microscope/Gao/250908-live-inhibitor/3-seg_cells')),
]
SAVE_DEBUG_IMAGES = False  # Threshold-sensitivity notebook: omit large debug TIFF outputs

# Imaging and detection parameters
PIXEL_SIZE_XY_NM    = 130.0
SPOT_RADIUS_XY_NM   = 200.0
THRESH_MCP  = 1.25
THRESH_TETR = 1.10
MAX_DISPLACEMENT_PX = 3.0
MIN_TRACK_CONSEC    = 3
DELTA_MIN           = 1.0
TETR_RESCUE_RADIUS_PX = 6
TETR_RESCUE_THR       = THRESH_TETR * 0.85
USE_TETR_RESCUE       = False   # initial default only; the driver loop at the BOTTOM of this cell overrides this and runs BOTH False and True

# [FIX] Max number of frames per movie (used for the dwell-time bound check).
#       None = infer automatically from each movie. If all Fig.5 movies are fixed at 60 frames,
#       set EXPECTED_T = 60 for a stricter hard-cap check.
EXPECTED_T = None   # movies may be 60 OR 61 frames; None -> dwell bound is inferred per-run from the data (safe, never false-alarms)

# Define a fixed order for plotting groups
GROUP_ORDER = ["DMSO", "TSA", "RGFP966"]


# BigFISH import (Requires installation: pip install big-fish)

try:
    import bigfish.detection as bf_detect
except ImportError as e:
    raise ImportError("bigfish not found. Please install with: pip install big-fish") from e


# Helper Functions (Identical to original script logic)


def gaussian_2d(xy, amp, x0, y0, sx, sy, theta):
    x, y = xy
    x0, y0 = float(x0), float(y0)
    a = (np.cos(theta)**2)/(2*sx**2) + (np.sin(theta)**2)/(2*sy**2)
    b = -(np.sin(2*theta))/(4*sx**2) + (np.sin(2*theta))/(4*sy**2)
    c = (np.sin(theta)**2)/(2*sx**2) + (np.cos(theta)**2)/(2*sy**2)
    g = amp * np.exp( - (a*((x-x0)**2) + 2*b*(x-x0)*(y-y0) + c*((y-y0)**2)))
    return g.ravel()

def measure_spot_intensity_gaussian(img: np.ndarray, yx: Tuple[float, float], patch_size: int = 9) -> float:
    """Fits a 2D Gaussian on (already normalized) image patch; returns amplitude-integrated intensity (relative units)."""
    if patch_size % 2 == 0: patch_size += 1
    h, w = img.shape
    y, x = int(round(yx[0])), int(round(yx[1]))
    s = patch_size // 2

    if y - s < 0 or y + s + 1 > h or x - s < 0 or x + s + 1 > w:
        return np.nan

    patch = img[y-s : y+s+1, x-s : x+s+1].astype(np.float64)
    Y, X = np.mgrid[:patch_size, :patch_size]
    amp0 = np.max(patch)
    initial_guess = (max(amp0, 0.0), s, s, 1.5, 1.5, 0.0)
    try:
        popt, _ = curve_fit(gaussian_2d, (X.ravel(), Y.ravel()), patch.ravel(), p0=initial_guess,
                            bounds=([0, 0, 0, 0.5, 0.5, -np.pi], [np.inf, patch_size, patch_size, s, s, np.pi]))
        amp, x0, y0, sx, sy, _ = popt
        intensity = 2 * np.pi * amp * sx * sy
        return intensity if np.isfinite(intensity) else np.nan
    except (RuntimeError, ValueError):
        return np.nan

def robust_bg_norm(img2d: np.ndarray) -> Tuple[np.ndarray, float, float]:
    """Robust background normalization using median absolute deviation (fallback without label)"""
    flat = img2d.reshape(-1)
    med = np.median(flat)
    mad = np.median(np.abs(flat - med))
    bg_std = (mad / 0.6745) if mad > 0 else max(np.std(flat), 1e-6)
    inorm = (img2d - med) / max(bg_std, 1e-6)
    return inorm.astype(np.float32), float(med), float(bg_std)

def nucleus_norm_image(img2d: np.ndarray, labels_2d: Optional[np.ndarray], cell_id: Optional[int]) -> Tuple[np.ndarray, float, float]:
    """Normalizes by 'intranuclear mean/robust SD'; if no label, falls back to global robust normalization."""
    if labels_2d is None or cell_id is None or cell_id <= 0 or (labels_2d==cell_id).sum() == 0:
        return robust_bg_norm(img2d)
    mask = (labels_2d == cell_id)
    vals = img2d[mask].astype(np.float64)
    if vals.size < 10:
        return robust_bg_norm(img2d)
    mu  = float(np.mean(vals))
    mad = float(np.median(np.abs(vals - mu)))
    sigma = (mad / 0.6745) if mad > 0 else max(float(np.std(vals)), 1e-6)
    inorm = (img2d - mu) / max(sigma, 1e-6)
    return inorm.astype(np.float32), mu, sigma

def detect_spots_norm(inorm: np.ndarray, thr: float) -> np.ndarray:
    """Detect spots using BigFISH (threshold detection on normalized image)"""
    spots = bf_detect.detect_spots(
        images=inorm, threshold=thr, return_threshold=False,
        voxel_size=(PIXEL_SIZE_XY_NM, PIXEL_SIZE_XY_NM),
        spot_radius=(SPOT_RADIUS_XY_NM, SPOT_RADIUS_XY_NM)
    )
    return spots.astype(np.float32) if spots is not None else np.empty((0, 2), dtype=np.float32)

def classify_group(stem: str) -> str:
    if stem.startswith("1-"): return "DMSO"
    if stem.startswith("2-"): return "TSA"
    if stem.startswith("3-"): return "RGFP966"
    return "Unknown"

def load_label_stack_for(tiff_path: Path):
    lab_path = tiff_path.with_name(f"{tiff_path.stem}_labels.tif")
    if not lab_path.exists(): return None
    labs = tifffile.imread(str(lab_path))
    if labs.ndim != 3: raise ValueError(f"Labels shape {labs.shape} expected (T,Y,X)")
    return labs.astype(np.int32)

def split_spots_by_cell(spots_yx: np.ndarray, labels_2d: np.ndarray) -> Dict[int, np.ndarray]:
    out = {}
    H, W = labels_2d.shape
    for y, x in spots_yx:
        yi, xi = int(np.clip(round(y), 0, H-1)), int(np.clip(round(x), 0, W-1))
        cid = int(labels_2d[yi, xi])
        if cid > 0:
            out.setdefault(cid, []).append([float(y), float(x)])
    return {k: np.asarray(v, dtype=np.float32) for k, v in out.items()}

def find_closest_spot(spots: np.ndarray, ref_yx: Tuple[float, float], max_dist: float) -> Optional[Tuple[float, float]]:
    if spots is None or len(spots) == 0 or ref_yx is None: return None
    dists = np.linalg.norm(spots - np.array(ref_yx), axis=1)
    min_idx = np.argmin(dists)
    return tuple(spots[min_idx]) if dists[min_idx] <= max_dist else None

def select_single_spot_per_cell(tetr_spots: np.ndarray, mcp_spots: np.ndarray, prev_tetr_yx=None, prev_mcp_yx=None):
    selected_tetr, selected_mcp = None, None
    if prev_tetr_yx: selected_tetr = find_closest_spot(tetr_spots, prev_tetr_yx, MAX_DISPLACEMENT_PX)
    if prev_mcp_yx: selected_mcp = find_closest_spot(mcp_spots, prev_mcp_yx, MAX_DISPLACEMENT_PX)
    if selected_tetr and not selected_mcp: selected_mcp = find_closest_spot(mcp_spots, selected_tetr, MAX_DISPLACEMENT_PX)
    elif selected_mcp and not selected_tetr: selected_tetr = find_closest_spot(tetr_spots, selected_mcp, MAX_DISPLACEMENT_PX)
    if not selected_tetr and not selected_mcp and len(tetr_spots) > 0 and len(mcp_spots) > 0:
        dists = np.linalg.norm(tetr_spots[:, np.newaxis, :] - mcp_spots[np.newaxis, :, :], axis=2)
        min_dist = np.min(dists)
        if min_dist <= MAX_DISPLACEMENT_PX:
            t_idx, m_idx = np.unravel_index(np.argmin(dists), dists.shape)
            selected_tetr, selected_mcp = tuple(tetr_spots[t_idx]), tuple(mcp_spots[m_idx])
    if not selected_tetr and not prev_tetr_yx and len(tetr_spots) > 0 and not selected_mcp: selected_tetr = tuple(tetr_spots[0])
    if not selected_mcp and not prev_mcp_yx and not prev_tetr_yx and len(mcp_spots) > 0 and not selected_tetr: selected_mcp = tuple(mcp_spots[0])
    return selected_tetr, selected_mcp

def extract_runs_with_censoring(binary: np.ndarray) -> Dict[str, List]:
    if len(binary) == 0: return dict(on_lengths=[], on_left_censored=[], on_right_censored=[], off_lengths=[], off_left_censored=[], off_right_censored=[])
    T = len(binary)
    runs, cur, start = [], binary[0], 0
    for i in range(1, T):
        if binary[i] != cur:
            runs.append((cur, start, i-1)); cur, start = binary[i], i
    runs.append((cur, start, T-1))
    on_L, on_lc, on_rc, off_L, off_lc, off_rc = [], [], [], [], [], []
    for v, s, e in runs:
        L, lc, rc = e - s + 1, (s == 0), (e == T-1)
        (on_L if v == 1 else off_L).append(L); (on_lc if v == 1 else off_lc).append(lc); (on_rc if v == 1 else off_rc).append(rc)
    return dict(on_lengths=on_L, on_left_censored=on_lc, on_right_censored=on_rc, off_lengths=off_L, off_left_censored=off_lc, off_right_censored=off_rc)

def nll_discrete_exp(lmbda: float, lengths: List[int], cens: List[bool], delta: float) -> float:
    if lmbda <= 0: return np.inf
    ll = 0.0
    for k, is_cens in zip(lengths, cens):
        if k <= 0: continue
        ll += np.log(np.exp(-lmbda*delta*(k-1)) + 1e-300) if is_cens else np.log(np.exp(-lmbda*delta*(k-1)) - np.exp(-lmbda*delta*k) + 1e-300)
    return -ll

def mle_rate_discrete(lengths: List[int], left_cens: List[bool], right_cens: List[bool], delta: float) -> Tuple[float, Dict]:
    if not lengths: return np.nan, {"n_exact":0, "n_cens":0}
    C = [lc or rc for lc, rc in zip(left_cens, right_cens)]
    L_exact = [k for k, c in zip(lengths, C) if not c]
    ub = 10.0
    if L_exact and np.mean(L_exact) > 0: ub = max(2.0 / (np.mean(L_exact) * delta), 1.0)
    res = minimize_scalar(lambda lam: nll_discrete_exp(lam, lengths, C, delta), bounds=(1e-6, ub), method="bounded")
    return float(res.x), {"n_exact": int(sum(not c for c in C)), "n_cens": int(sum(c for c in C))}

def burst_freq(Ton: float, Toff: float) -> float:
    if np.any(np.isnan([Ton, Toff])): return np.nan
    s = Ton + Toff
    return (1.0 / s) if s > 0 else np.nan

def rescue_spot_from_window(inorm: np.ndarray,
                            center_yx: Tuple[float, float],
                            radius_px: int,
                            thr: float) -> Optional[Tuple[float, float]]:
    """Local max search in window around center; return (y,x) if max >= thr."""
    if center_yx is None:
        return None
    H, W = inorm.shape
    cy, cx = int(round(center_yx[0])), int(round(center_yx[1]))
    y0, y1 = max(0, cy - radius_px), min(H, cy + radius_px + 1)
    x0, x1 = max(0, cx - radius_px), min(W, cx + radius_px + 1)
    if y0 >= y1 or x0 >= x1:
        return None
    patch = inorm[y0:y1, x0:x1]
    maxv = float(np.max(patch))
    if maxv < thr:
        return None
    iy, ix = np.unravel_index(np.argmax(patch), patch.shape)
    return (float(y0 + iy), float(x0 + ix)) #####added


# [FIX] ---------------------------------------------------------------------
# Shared run extractor: extract ON/OFF runs from a frame-sorted 0/1 sequence.
# Reused by both aggregated_runs.csv and runs_by_cell to avoid two inconsistent implementations.
def extract_segments_sorted(binary: np.ndarray) -> List[dict]:
    binary = np.asarray(binary).astype(int)
    T = len(binary)
    if T == 0:
        return []
    segs, cur, start = [], binary[0], 0
    for i in range(1, T):
        if binary[i] != cur:
            segs.append((cur, start, i - 1)); cur, start = binary[i], i
    segs.append((cur, start, T - 1))
    out = []
    for state, s, e in segs:
        out.append({
            "state": int(state),
            "start_frame": int(s),
            "end_frame": int(e),
            "length_frames": int(e - s + 1),
            "left_censored": (s == 0),
            "right_censored": (e == T - 1),
        })
    return out


def dedup_perframe(perframes: pd.DataFrame, tag: str = "") -> pd.DataFrame:
    """[FIX] Drop duplicates on (file, cell_id, frame) and check per-cell frame uniqueness."""
    n0 = len(perframes)
    perframes = perframes.drop_duplicates(subset=["file", "cell_id", "frame"], keep="first").copy()
    n_drop = n0 - len(perframes)
    if n_drop > 0:
        print(f"  [QC][WARN]{tag} dropped {n_drop} duplicated (file,cell_id,frame) rows "
              f"(trace of the old glob self-inclusion bug; delete stale aggregated_*.csv in the output folder and rerun)")
      # Frames must be unique within each cell
    g = perframes.groupby(["file", "cell_id"])["frame"]
    if not (g.transform("nunique") == g.transform("size")).all():
        raise ValueError(f"{tag} duplicate frames remain within a (file, cell_id) after dedup")
    return perframes
# ---------------------------------------------------------------------------


# Core: Process a single TIFF

def process_tiff_file(tiff_path: Path, out_dir: Path) -> None:
    """Processes a single TIFF file and saves per-file CSV results into out_dir."""
    arr = tifffile.imread(str(tiff_path))
    if arr.ndim != 4 or arr.shape[1] != 2: raise ValueError(f"TIFF shape {arr.shape} not (T, 2, Y, X).")
    T, C, Y, X = arr.shape
    MCP, TETR = 0, 1
    labels = load_label_stack_for(tiff_path)
    if labels is not None and labels.shape[0] != T: raise ValueError(f"Label T mismatch: {labels.shape[0]} vs {T}")

    mcp_raw_cache, tetr_raw_cache, mcp_spots_cache, tetr_spots_cache = [], [], [], []
    for t in range(T):
        mcp_raw, tetr_raw = arr[t, MCP].astype(np.float32), arr[t, TETR].astype(np.float32)
        mcp_norm, _, _ = robust_bg_norm(mcp_raw)
        tetr_norm, _, _ = robust_bg_norm(tetr_raw)
        mcp_raw_cache.append(mcp_raw)
        tetr_raw_cache.append(tetr_raw)
        mcp_spots_cache.append(detect_spots_norm(mcp_norm, THRESH_MCP))
        tetr_spots_cache.append(detect_spots_norm(tetr_norm, THRESH_TETR))

    cell_ids = [1]
    if labels is not None: cell_ids = sorted({int(x) for t in range(T) for x in np.unique(labels[t]) if x > 0})

    cell_selected_spots = {cid: {'tetr': {}, 'mcp': {}} for cid in cell_ids}
    for cid in cell_ids:
        prev_tetr, prev_mcp = None, None
        for t in range(T):
            labs2d = labels[t] if labels is not None else np.ones((Y,X), dtype=int)
            tetr_in_cell = split_spots_by_cell(tetr_spots_cache[t], labs2d).get(cid, np.empty((0,2)))
            mcp_in_cell = split_spots_by_cell(mcp_spots_cache[t], labs2d).get(cid, np.empty((0,2)))
            sel_tetr, sel_mcp = select_single_spot_per_cell(tetr_in_cell, mcp_in_cell, prev_tetr, prev_mcp)
            if sel_tetr: cell_selected_spots[cid]['tetr'][t] = sel_tetr; prev_tetr = sel_tetr
            if sel_mcp: cell_selected_spots[cid]['mcp'][t] = sel_mcp; prev_mcp = sel_mcp

    for cid in cell_ids:
        prev_tetr = None
        for t in range(T):
            tetr_yx = cell_selected_spots[cid]['tetr'].get(t, None)
            mcp_yx  = cell_selected_spots[cid]['mcp'].get(t, None)
            labs2d = labels[t] if labels is not None else None

            tetr_norm_cell, _, _ = nucleus_norm_image(tetr_raw_cache[t], labs2d, cid)
            if tetr_yx is None and USE_TETR_RESCUE:

                center = mcp_yx if mcp_yx is not None else prev_tetr
                tetr_yx = rescue_spot_from_window(
                    tetr_norm_cell, center_yx=center,
                    radius_px=TETR_RESCUE_RADIUS_PX,
                    thr=TETR_RESCUE_THR
                )

                if tetr_yx is None and (mcp_yx is not None) and (prev_tetr is not None):
                    tetr_yx = rescue_spot_from_window(
                        tetr_norm_cell, center_yx=prev_tetr,
                        radius_px=TETR_RESCUE_RADIUS_PX,
                        thr=TETR_RESCUE_THR
                    )
            if tetr_yx is not None:
                cell_selected_spots[cid]['tetr'][t] = tetr_yx
                prev_tetr = tetr_yx
            else:

                if prev_tetr is not None:
                    prev_tetr = prev_tetr  ###added

    rows = []
    for cid in cell_ids:
        for t in range(T):
            tetr_yx, mcp_yx = cell_selected_spots[cid]['tetr'].get(t), cell_selected_spots[cid]['mcp'].get(t)
            on_flag = 0
            if tetr_yx and mcp_yx and np.linalg.norm(np.array(tetr_yx) - np.array(mcp_yx)) <= MAX_DISPLACEMENT_PX: on_flag = 1
            elif mcp_yx: on_flag = 1
            rel_intensity = np.nan
            if on_flag and mcp_yx:
                labs2d = labels[t] if labels is not None else None
                mcp_norm_cell, _, _ = nucleus_norm_image(mcp_raw_cache[t], labs2d, cid)
                rel_intensity = measure_spot_intensity_gaussian(mcp_norm_cell, mcp_yx)
            rows.append(dict(file=tiff_path.name, group=classify_group(tiff_path.name), frame=t, cell_id=cid, on_off=on_flag, tetr_stable=bool(tetr_yx), mcp_y=mcp_yx[0] if mcp_yx else np.nan, mcp_x=mcp_yx[1] if mcp_yx else np.nan, mcp_integrated_intensity=rel_intensity))
    perframe = pd.DataFrame(rows).sort_values(["cell_id", "frame"]).reset_index(drop=True)

    final_dfs = []
    for cid, dfc in perframe.groupby("cell_id"):
        dfc = dfc.copy(); on_off = dfc['on_off'].values.copy()
        for i in range(len(dfc)):
            if not dfc['tetr_stable'].iloc[i] and dfc['on_off'].iloc[i] == 1:
                connected = (i > 0 and dfc['on_off'].iloc[i-1] == 1 and np.linalg.norm(dfc[['mcp_y','mcp_x']].iloc[i] - dfc[['mcp_y','mcp_x']].iloc[i-1]) <= MAX_DISPLACEMENT_PX) or \
                            (i < len(dfc)-1 and dfc['on_off'].iloc[i+1] == 1 and np.linalg.norm(dfc[['mcp_y','mcp_x']].iloc[i] - dfc[['mcp_y','mcp_x']].iloc[i+1]) <= MAX_DISPLACEMENT_PX)
                if not connected: on_off[i] = 0
        dfc['on_off'] = on_off
        final_dfs.append(dfc)
    if final_dfs: perframe = pd.concat(final_dfs, ignore_index=True)
    perframe.drop(columns=['mcp_y', 'mcp_x']).pipe(export_transcription_states).to_csv(out_dir / f"{tiff_path.stem}_perframe.csv", index=False)

    if SAVE_DEBUG_IMAGES:
        debug_stack = []
        for t in range(T):
            mcp_draw, tetr_draw = arr[t, MCP].copy(), arr[t, TETR].copy()
            mcp_v, tetr_v = np.percentile(mcp_draw, 99.9), np.percentile(tetr_draw, 99.9)
            for cid in cell_ids:
                if t in cell_selected_spots[cid]['tetr']: rr, cc = circle_perimeter(int(round(cell_selected_spots[cid]['tetr'][t][0])), int(round(cell_selected_spots[cid]['tetr'][t][1])), 5, shape=(Y,X)); tetr_draw[rr, cc] = tetr_v
                if t in cell_selected_spots[cid]['mcp']: rr, cc = circle_perimeter(int(round(cell_selected_spots[cid]['mcp'][t][0])), int(round(cell_selected_spots[cid]['mcp'][t][1])), 5, shape=(Y,X)); mcp_draw[rr, cc] = mcp_v
            debug_stack.append(np.stack([mcp_draw, tetr_draw], axis=0))
        tifffile.imsave(out_dir / f"{tiff_path.stem}_spots_debug.tif", np.array(debug_stack, dtype=np.float32), imagej=True)

    kin_rows = []
    for cid, dfc in perframe.groupby("cell_id", sort=True):
        if dfc['on_off'].sum() == 0 or len(dfc) < MIN_TRACK_CONSEC * 2: continue
            # [FIX] Sort by frame before run extraction for kinetics too (explicit and safer)
        seq = dfc.sort_values("frame")["on_off"].to_numpy()
        rr = extract_runs_with_censoring(seq)
        b_hat, _ = mle_rate_discrete(rr["on_lengths"], rr["on_left_censored"], rr["on_right_censored"], DELTA_MIN)
        a_hat, _ = mle_rate_discrete(rr["off_lengths"], rr["off_left_censored"], rr["off_right_censored"], DELTA_MIN)
        Ton = (1.0 / b_hat) if (b_hat > 0 and np.isfinite(b_hat)) else np.nan
        Toff = (1.0 / a_hat) if (a_hat > 0 and np.isfinite(a_hat)) else np.nan
        kin_rows.append(dict(file=tiff_path.name, group=classify_group(tiff_path.name), cell_id=cid, a_off_to_on=a_hat, b_on_to_off=b_hat, T_on_mean=Ton, T_off_mean=Toff, burst_frequency_per_min=burst_freq(Ton, Toff), frames_analyzed=len(seq)))
    if kin_rows: pd.DataFrame(kin_rows).pipe(export_transcription_states).to_csv(out_dir / f"{tiff_path.stem}_kinetics.csv", index=False)

def run_analysis_and_save_results(root_dir: str, gene: str):
    """Main function: Runs all analyses and saves aggregated results to CSV."""
    root   = Path(root_dir)
    suffix = "" if USE_TETR_RESCUE else "_standard_only"
    outdir = OUTPUTS_DIR / gene / f"tetr_mcp_hybrid_analysis{suffix}"
    outdir.mkdir(parents=True, exist_ok=True)

    # [FIX] Remove stale aggregate files before rerunning, to prevent glob self-inclusion at the source.
    for stale in outdir.glob("aggregated_*.csv"):
        stale.unlink()
        print(f"  [QC] removed stale aggregate: {stale.name}")

    files = sorted([p for p in root.glob("*.tif") if not p.name.endswith("_labels.tif")])
    print(f"Starting processing of {len(files)} TIFF files...")
    for i, fp in enumerate(files, 1):
        print(f"  [{i}/{len(files)}] Processing {fp.name}...")
        try:
            process_tiff_file(fp, outdir)
        except Exception as e:
            print(f"  Error processing {fp.name}: {e}")

    print("\nAggregating all results...")
    # [FIX] Exclude the aggregate outputs themselves so aggregated_perframe.csv / aggregated_kinetics.csv are not re-read.
    perframe_files = [p for p in outdir.glob("*_perframe.csv") if not p.name.startswith("aggregated")]
    kinetics_files = [p for p in outdir.glob("*_kinetics.csv") if not p.name.startswith("aggregated")]

    if not perframe_files:
        print("No per-frame result files found. Exiting.")
        return None

    # 1. Aggregate per-frame data
    perframes = pd.concat([read_legacy_state_csv(f) for f in perframe_files], ignore_index=True)
    # [FIX] Dedup + frame-uniqueness check (a second safeguard even though the glob is fixed)
    perframes = dedup_perframe(perframes, tag="[perframe]")
    perframes = perframes.sort_values(["group", "file", "cell_id", "frame"]).reset_index(drop=True)
    perframes.pipe(export_transcription_states).to_csv(outdir / "aggregated_perframe.csv", index=False)
    print(f"  Saved aggregated per-frame data to: {outdir / 'aggregated_perframe.csv'}")

    # 2. Aggregate kinetic data
    kinetics = pd.DataFrame()
    if kinetics_files:
        kinetics = pd.concat([read_legacy_state_csv(f) for f in kinetics_files if f.stat().st_size > 0], ignore_index=True)
        # [FIX] kinetics has one row per (file, cell_id); dedup to prevent double aggregation
        if not kinetics.empty:
            kinetics = kinetics.drop_duplicates(subset=["file", "cell_id"], keep="first")
    kinetics.pipe(export_transcription_states).to_csv(outdir / "aggregated_kinetics.csv", index=False)
    print(f"  Saved aggregated kinetic data to: {outdir / 'aggregated_kinetics.csv'}")

    # 3. Calculate and save mean intensities
    on_frames = perframes[perframes['on_off'] == 1].copy()
    if not on_frames.empty:
        cell_intensities = on_frames.groupby(['group', 'file', 'cell_id'])['mcp_integrated_intensity'].mean().reset_index()
        cell_intensities.rename(columns={'mcp_integrated_intensity': 'mean_intensity'}, inplace=True)
    else:
        cell_intensities = pd.DataFrame(columns=['group', 'file', 'cell_id', 'mean_intensity'])
    cell_intensities.pipe(export_transcription_states).to_csv(outdir / "aggregated_cell_intensities.csv", index=False)
    print(f"  Saved aggregated mean intensity data to: {outdir / 'aggregated_cell_intensities.csv'}")

    # 4. Extract and save all ON/OFF runs
    # [FIX] Use the shared sorted extractor; group by (group, file, cell_id) and sort by frame explicitly,
    #       to avoid the cross-boundary merging / censoring mislabeling caused by the old unsorted code.
    all_runs = []
    for (group, file, cell_id), dfc in perframes.groupby(['group', 'file', 'cell_id']):
        if len(dfc) < MIN_TRACK_CONSEC: continue
        seq = dfc.sort_values("frame")["on_off"].to_numpy()
        T_cell = len(seq)
        for seg in extract_segments_sorted(seq):
            # [FIX] dwell-time hard-cap check: a run cannot be longer than this cell's frame count
            assert seg["length_frames"] <= T_cell, \
                f"run length {seg['length_frames']} > T_cell {T_cell} ({file}, cell {cell_id})"
            all_runs.append({
                'group': group,
                'file': file,
                'cell_id': cell_id,
                'kind': 'ON' if seg["state"] == 1 else 'OFF',
                'length_frames': int(seg["length_frames"]),
                'length_min': float(seg["length_frames"]) * DELTA_MIN,
                'left_censored': bool(seg["left_censored"]),
                'right_censored': bool(seg["right_censored"]),
            })
    runs_all = pd.DataFrame(all_runs)
    runs_all.pipe(export_transcription_states).to_csv(outdir / "aggregated_runs.csv", index=False)
    print(f"  Saved aggregated ON/OFF run data to: {outdir / 'aggregated_runs.csv'}")

    print("\n=== Analysis Complete ===")
    print(f"All results are saved in: {outdir}")
    return outdir

def build_runs_by_cell(outdir: Path, delta_min: float):
    outdir = Path(outdir)
    PERFRAME_PATH = outdir / "aggregated_perframe.csv"

    if not PERFRAME_PATH.exists():
        raise FileNotFoundError(f"Not found: {PERFRAME_PATH} (run Cell 1 first)")

    perframes = read_legacy_state_csv(PERFRAME_PATH)

    required = {"group","file","cell_id","frame","on_off"}
    missing = required - set(perframes.columns)
    if missing:
        raise ValueError(f"aggregated_perframe.csv missing columns: {missing}")

     # [FIX] Dedup again after loading so runs_by_cell is never built from duplicated rows.
    perframes = dedup_perframe(perframes, tag="[runs_by_cell]")

    perframes["cell_uid"] = perframes["file"].astype(str) + "_c" + perframes["cell_id"].astype(str)
    perframes = perframes.sort_values(["group","cell_uid","frame"]).reset_index(drop=True)

    rows = []
    for (grp, cell_uid), dfc in perframes.groupby(["group","cell_uid"], sort=False):
        dfc = dfc.sort_values("frame")
        seq = dfc["on_off"].astype(int).to_numpy()
        T_cell = len(seq)
        segs = extract_segments_sorted(seq)
        file_ = dfc["file"].iloc[0]
        cell_id_ = dfc["cell_id"].iloc[0]
        for seg in segs:

            assert seg["length_frames"] <= T_cell, \
                f"run length {seg['length_frames']} > T_cell {T_cell} ({cell_uid})"
            rows.append({
                "group": grp,
                "file": file_,
                "cell_id": cell_id_,
                "cell_uid": cell_uid,
                "kind": "ON" if seg["state"] == 1 else "OFF",
                "start_frame": seg["start_frame"],
                "end_frame": seg["end_frame"],
                "length_frames": seg["length_frames"],
                "length_min": seg["length_frames"] * delta_min,
                "left_censored": bool(seg["left_censored"]),
                "right_censored": bool(seg["right_censored"]),
            })

    runs_cell = pd.DataFrame(rows)

    # [FIX] Global dwell-time bound check: no run may exceed the movie duration.
    #       This is the key assertion that directly addresses reviewer R3-M4 (KM curves beyond 60 min).
    if not runs_cell.empty:
        max_T = perframes.groupby("cell_uid")["frame"].nunique().max()
        movie_dur_min = (EXPECTED_T if EXPECTED_T is not None else max_T) * delta_min
        max_dwell = runs_cell["length_min"].max()
        assert max_dwell <= movie_dur_min + 1e-9, (
            f"dwell time {max_dwell} min exceeds movie duration {movie_dur_min} min "
            f"-> duplicated rows may remain or DELTA_MIN may be wrong; please check"
        )
        print(f"  [QC] max dwell = {max_dwell:.1f} min, movie duration = {movie_dur_min:.1f} min (OK)")

    # Save (use this for subsequent analyses)
    RUNS_CELL_PATH = outdir / "aggregated_runs_by_cell.csv"
    runs_cell.pipe(export_transcription_states).to_csv(RUNS_CELL_PATH, index=False)
    print(f"[saved] {RUNS_CELL_PATH}")
    return runs_cell


# ============================================================
# [R3-m11 SENSITIVITY] Run the FULL pipeline under BOTH center-calling
# configurations in a single pass, so the standard-vs-rescue comparison
# (Threshold-sensitivity QC cell) always finds BOTH output folders.
#
#   USE_TETR_RESCUE = True  -> outputs/<gene>/tetr_mcp_hybrid_analysis/
#                             == PAPER main-text result (rescue)
#   USE_TETR_RESCUE = False -> outputs/<gene>/tetr_mcp_hybrid_analysis_standard_only/
#                             == sensitivity check (standard threshold only)
#
# NOTE 1: the rescue step changes spot DETECTION itself, so the whole
#         per-frame pipeline is re-run for each config -> this cell runs
#         the analysis TWICE (expect ~2x runtime).
# NOTE 2: standard is run FIRST and rescue LAST on purpose, so the notebook
#         is left in the PAPER configuration (USE_TETR_RESCUE == True) when
#         this cell finishes.
# ============================================================
for _use_rescue in (False, True):
    USE_TETR_RESCUE = _use_rescue          # module-global; read by run_analysis_and_save_results / process_tiff_file
    _cfg = "rescue (PAPER)" if _use_rescue else "standard_only (sensitivity)"
    print(f"\n########## CONFIG: USE_TETR_RESCUE={_use_rescue}  [{_cfg}] ##########")
    for gene, root_dir in DATASETS:
        print(f"\n=== {gene} | {_cfg} ===")
        outdir = run_analysis_and_save_results(root_dir, gene)
        if outdir is None:
            continue
        build_runs_by_cell(outdir, DELTA_MIN)

## Downstream visualizations (paper-ready panels)

This section generates:
- State raster plots (cells × time)
- Duty cycle (fraction of Active frames per cell)
- Kaplan–Meier curves for Active/Inactive durations
- QC summary tables (cells, runs, censoring fractions)

All plots are saved as PDF for direct use in figure assembly.


In [ ]:
# --- Imports & global plot style (Arial, PDF) ---

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# (optional) stats utilities
from scipy.stats import chi2

# ---- Output folders ----
if "OUTPUTS_DIR" not in globals():
    raise RuntimeError("Run the main analysis-definition cell first to define OUTPUTS_DIR.")
FIGDIR = Path(OUTPUTS_DIR) / "figs"
TABLEDIR = Path(OUTPUTS_DIR) / "tables"
FIGDIR.mkdir(exist_ok=True, parents=True)
TABLEDIR.mkdir(exist_ok=True, parents=True)

# ---- Global plotting parameters (edit here) ----
PLOT_CFG = {
    # font sizes
    "font_base": 8,
    "font_axes_label": 9,
    "font_axes_title": 9,
    "font_ticks": 8,
    "font_legend": 7,

    # figure sizes (inches)
    "figsize_small": (3.3, 2.6),
    "figsize_medium": (4.0, 3.0),
    "figsize_wide": (6.0, 3.0),
    "dpi": 300,
}

def set_mpl_style(cfg=PLOT_CFG):
    """Set matplotlib rcParams (Arial + PDF text embedding)."""
    plt.rcParams.update({
        # font
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": cfg["font_base"],
        "axes.labelsize": cfg["font_axes_label"],
        "axes.titlesize": cfg["font_axes_title"],
        "xtick.labelsize": cfg["font_ticks"],
        "ytick.labelsize": cfg["font_ticks"],
        "legend.fontsize": cfg["font_legend"],

        # publication-friendly PDF/PS fonts (editable text)
        "pdf.fonttype": 42,
        "ps.fonttype": 42,

        # simple look
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

set_mpl_style()
print("[OK] matplotlib style set. Output:", FIGDIR.resolve())


In [ ]:
# --- Paths to precomputed analysis outputs (edit if needed) ---

# Read analysis outputs from the local output root defined above.
if "OUTPUTS_DIR" not in globals():
    raise RuntimeError("Run the main analysis-definition cell first to define OUTPUTS_DIR.")
OUTPUTS_DIR = Path(OUTPUTS_DIR)

NANOG_DIR = (OUTPUTS_DIR / "Nanog" / "tetr_mcp_hybrid_analysis").resolve()
SOX2_DIR  = (OUTPUTS_DIR / "Sox2"  / "tetr_mcp_hybrid_analysis").resolve()

print("NANOG_DIR:", NANOG_DIR)
print("SOX2_DIR :", SOX2_DIR)

for p in [NANOG_DIR, SOX2_DIR]:
    if not p.exists():
        print(f"[WARN] Not found: {p}")


In [ ]:
# --- Data loader (reuses aggregated CSVs) ---

def _ensure_cell_uid(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "cell_uid" not in df.columns:
        if ("file" in df.columns) and ("cell_id" in df.columns):
            df["cell_uid"] = df["file"].astype(str) + "_c" + df["cell_id"].astype(str)
        else:
            raise ValueError("Cannot construct cell_uid (need columns: file, cell_id).")
    return df

def load_dataset(gene: str, analysis_dir: Path) -> dict:
    analysis_dir = Path(analysis_dir)

    perframe_path = analysis_dir / "aggregated_perframe.csv"
    runs_by_cell_path = analysis_dir / "aggregated_runs_by_cell.csv"
    runs_path = analysis_dir / "aggregated_runs.csv"

    if not perframe_path.exists():
        raise FileNotFoundError(perframe_path)

    perframe = read_legacy_state_csv(perframe_path)
    perframe = _ensure_cell_uid(perframe)

    if runs_by_cell_path.exists():
        runs = read_legacy_state_csv(runs_by_cell_path)
    elif runs_path.exists():
        runs = read_legacy_state_csv(runs_path)
    else:
        raise FileNotFoundError(f"Neither {runs_by_cell_path.name} nor {runs_path.name} exists in {analysis_dir}")

    runs = _ensure_cell_uid(runs)

    # Basic sanity checks
    req_runs = {"group","kind","length_min","left_censored","right_censored","cell_uid"}
    missing = req_runs - set(runs.columns)
    if missing:
        raise ValueError(f"Missing columns in runs: {missing}")

    req_pf = {"group","frame","on_off","cell_uid"}
    missing_pf = req_pf - set(perframe.columns)
    if missing_pf:
        raise ValueError(f"Missing columns in perframe: {missing_pf}")

    return {
        "gene": gene,
        "dir": analysis_dir,
        "perframe": perframe,
        "runs": runs,
    }

datasets = []
for gene, p in [("Nanog", NANOG_DIR), ("Sox2", SOX2_DIR)]:
    try:
        datasets.append(load_dataset(gene, p))
        print(f"[OK] Loaded {gene}: perframe={len(datasets[-1]['perframe'])}, runs={len(datasets[-1]['runs'])}")
    except Exception as e:
        print(f"[SKIP] {gene}: {e}")

datasets


In [ ]:
# --- Group labels / ordering ---

# Minimal (main-figure) set
GROUP_ORDER_MAIN = ["DMSO", "TSA", "RGFP966"]

# Full set (ED)
GROUP_ORDER_FULL = ["DMSO", "TSA", "RGFP966"]

DISPLAY_NAME = {
    "RGFP966": "RGFP966",
}


KIND_LABEL = {
    "ON": "Active",
    "OFF": "Inactive"
}

def nice_group(g: str) -> str:
    return DISPLAY_NAME.get(g, g)


In [ ]:
# --- Survival analysis utilities (KM) ---

def kaplan_meier(times, events):
    """Kaplan–Meier estimator for right-censored data.

    Parameters
    ----------
    times : array-like (>=0)
    events : array-like (1=event observed, 0=right-censored)

    Returns
    -------
    t : np.ndarray
        Step times, starting with 0.
    S : np.ndarray
        Survival values S(t)=P(T>=t) in 'post' convention.
    """
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)

    m = np.isfinite(times) & (times >= 0) & np.isfinite(events)
    times = times[m]; events = events[m]

    if len(times) == 0:
        return np.array([0.0]), np.array([1.0])

    event_times = np.unique(times[events == 1])
    S = 1.0
    t_list = [0.0]
    s_list = [1.0]

    for t in event_times:
        n_risk = np.sum(times >= t)
        d = np.sum((times == t) & (events == 1))
        if n_risk <= 0:
            continue
        S *= (1.0 - d / n_risk)
        t_list.append(float(t))
        s_list.append(float(S))

    return np.array(t_list), np.array(s_list)

def prepare_runs(runs: pd.DataFrame, kind: str, groups, exclude_left=True):
    df = runs[(runs["kind"] == kind) & (runs["group"].isin(groups))].copy()
    if exclude_left:
        df = df[~df["left_censored"]].copy()
    df["event"] = (~df["right_censored"]).astype(int)
    df = df[np.isfinite(df["length_min"]) & (df["length_min"] > 0)].copy()
    return df

In [ ]:
# --- Plot helpers (save as PDF) ---

def savefig_pdf(fig, outpath: Path):
    outpath = Path(outpath)
    outpath.parent.mkdir(exist_ok=True, parents=True)
    fig.savefig(outpath, bbox_inches="tight", dpi=PLOT_CFG["dpi"])
    plt.close(fig)
    print("[saved]", outpath)

def plot_km_panel(runs: pd.DataFrame, kind: str, gene: str, groups, exclude_left=True,
                  logy=True, xlim=None, ylim=None, title=None, outname=None):

    df = prepare_runs(runs, kind=kind, groups=groups, exclude_left=exclude_left)

    fig, ax = plt.subplots(figsize=(3.33,2.5))

    for grp in groups:
        sub = df[df["group"] == grp]
        if sub.empty:
            continue
        t, S = kaplan_meier(sub["length_min"].values, sub["event"].values)
        ax.step(t, S, where="post",
                label=f"{nice_group(grp)} (runs={len(sub)})")

    # ---- Convert labels for display ----
    kind_label = KIND_LABEL.get(kind, kind)

    ax.set_xlabel(f"{kind_label} duration (min)")
    ax.set_ylabel("Survival S(t) = P(duration ≥ t)")

    if title is None:
        title = f"{gene}: Kaplan–Meier ({kind_label})"
    ax.set_title(title)

    if logy:
        ax.set_yscale("log")
        if ylim is None:
            ax.set_ylim(1e-3, 1.05)
    else:
        ax.set_ylim(0, 1.05)

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    ax.grid(alpha=0.3)
    ax.legend(frameon=False)

    if outname is None:
        outname = f"{gene}_KM_{kind_label}_groups{'-'.join(groups)}.pdf"
    savefig_pdf(fig, FIGDIR / outname)






from scipy.stats import mannwhitneyu
import numpy as np

def holm_adjust_array(pvals):
    """Holm step-down adjustment for a list/array of p-values."""
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m, dtype=float)

    running_max = 0.0
    for rank, idx in enumerate(order):
        adj_p = (m - rank) * pvals[idx]
        running_max = max(running_max, adj_p)
        adj[idx] = min(1.0, running_max)
    return adj

def p_to_stars(p):
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 5e-2: return "*"
    return "ns"

def add_sig_bracket(ax, x1, x2, y, text, h=0.02, lw=0.8):
    """
    Draw a significance bracket between x1 and x2 at height y (data coords).
    h: vertical height of bracket (data coords)
    """
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], color="black", lw=lw, clip_on=False)
    ax.text((x1+x2)/2, y+h, text, ha="center", va="bottom", fontsize=PLOT_CFG.get("fontsize", 7))



def duty_cycle_pvals(dc: pd.DataFrame, base="DMSO", comps=("TSA","RGFP966")):
    """
    dc: output of duty_cycle_table (columns: group, cell_uid, duty_cycle)
    returns: list of dict with raw p and Holm-adjusted p
    """
    base_vals = dc.loc[dc["group"]==base, "duty_cycle"].dropna().values
    results = []

    raw_ps = []
    pairs = []
    for g in comps:
        vals = dc.loc[dc["group"]==g, "duty_cycle"].dropna().values
        if len(base_vals) < 2 or len(vals) < 2:
            p = np.nan
        else:
            p = mannwhitneyu(base_vals, vals, alternative="two-sided").pvalue
        raw_ps.append(p)
        pairs.append((base, g))

    adj_ps = holm_adjust_array(raw_ps)

    for (b, g), p, p_adj in zip(pairs, raw_ps, adj_ps):
        results.append({
            "compare": f"{b} vs {g}",
            "p_raw": float(p) if np.isfinite(p) else np.nan,
            "p_holm": float(p_adj) if np.isfinite(p_adj) else np.nan,
        })
    return results



def duty_cycle_table(perframe: pd.DataFrame, groups) -> pd.DataFrame:
    """
    perframe: expects at least {group, cell_uid, on_off} in columns
    duty_cycle = mean on_off per cell (= fraction of ON frames)
    """
    pf = perframe[perframe["group"].isin(groups)].copy()
    pf = _ensure_cell_uid(pf)

    # Treat on_off as numeric (assumed 0/1)
    pf = pf[np.isfinite(pf["on_off"])].copy()

    dc = (pf.groupby(["group", "cell_uid"], sort=False)["on_off"]
            .mean()
            .reset_index()
            .rename(columns={"on_off": "duty_cycle"}))
    return dc


def plot_duty_cycle(perframe: pd.DataFrame, gene: str, groups, outname=None, title=None):
    dc = duty_cycle_table(perframe, groups)
    fig, ax = plt.subplots(figsize=(1.2,2.3))

    data = [dc.loc[dc["group"]==g, "duty_cycle"].values for g in groups]
    ax.boxplot(data, positions=np.arange(len(groups)), widths=0.5, showfliers=False)

    # overlay points (jitter)
    rng = np.random.default_rng(0)
    for i, g in enumerate(groups):
        y = dc.loc[dc["group"]==g, "duty_cycle"].values
        x = rng.normal(i, 0.06, size=len(y))
        ax.plot(x, y, "o", markersize=3, alpha=0.6)

    # -------------------- p-value annotations (MW + Holm) --------------------
    # Here, compare TSA/RGFP966 against DMSO as the base (only if present)
    base = "DMSO"
    comps = [g for g in ["TSA", "RGFP966"] if g in groups]

    if base in groups and len(comps) > 0:
        stats = duty_cycle_pvals(dc, base=base, comps=tuple(comps))

        # Bracket height: add a little above the data maximum
        y_max = np.nanmax(dc["duty_cycle"].values)
        y = y_max + 0.03  # Adjust here for appearance (duty is 0–1, so a fixed offset is OK)
        step = 0.12       # Stagger to avoid overlapping multiple brackets

        # Use the index within groups for x positions
        g_to_x = {g:i for i, g in enumerate(groups)}

        for k, r in enumerate(stats):
            comp = r["compare"]  # "DMSO vs TSA"
            g = comp.split(" vs ")[1]
            x1 = g_to_x[base]
            x2 = g_to_x[g]

            p_adj = r["p_holm"]
            # Display format: show p-values or stars
            label = f"{p_to_stars(p_adj)}" if np.isfinite(p_adj) else "n/a"
            # If you want to show p-values directly:
            # label = f"p={p_adj:.3g}" if np.isfinite(p_adj) else "n/a"

            add_sig_bracket(ax, x1, x2, y + k*step, label, h=0.015, lw=0.8)

    # ------------------------------------------------------------------------

    ax.set_xticks(np.arange(len(groups)))
    ax.set_xticklabels([nice_group(g) for g in groups], rotation=30, ha="right")
    ax.set_ylabel("Fraction of Active frames")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)

    if outname is None:
        outname = f"{gene}_duty_cycle_groups{'-'.join(groups)}.pdf"
    savefig_pdf(fig, FIGDIR / outname)
    return dc



def plot_state_raster(perframe: pd.DataFrame, gene: str, groups, n_cells=35,
                      frame_to_min=1.0, outname=None, title=None):
    """ON/OFF raster (cells × time) for each group, stacked vertically."""
    pf = perframe[perframe["group"].isin(groups)].copy()
    pf = _ensure_cell_uid(pf)

    # build matrices per group
    mats = []
    for g in groups:
        sub = pf[pf["group"]==g].copy()
        if sub.empty:
            mats.append(None)
            continue

        mat = sub.pivot_table(index="cell_uid", columns="frame", values="on_off", aggfunc="first")
        # sort cells by duty cycle (descending)
        mat = mat.loc[mat.mean(axis=1).sort_values(ascending=False).index]
        mat = mat.iloc[:n_cells]
        mats.append(mat)

    # determine common x-axis
    max_frame = 0
    for m in mats:
        if m is not None and m.shape[1] > max_frame:
            max_frame = m.shape[1]
    t = np.arange(max_frame) * frame_to_min

    fig_h = 0.8*len(groups) + 1.2
    #fig, axes = plt.subplots(nrows=len(groups), ncols=1, figsize=(PLOT_CFG["figsize_wide"][0], fig_h), sharex=True)
    fig, axes = plt.subplots(nrows=len(groups), ncols=1, figsize=(2.0, 4.0), sharex=True)
    if len(groups) == 1:
        axes = [axes]

    for ax, g, mat in zip(axes, groups, mats):
        if mat is None:
            ax.axis("off")
            continue
        # reindex columns to full range
        mat2 = mat.reindex(columns=np.arange(max_frame))
        from matplotlib.colors import ListedColormap

        cmap = ListedColormap(["lightgray", "red"])  # OFF=0, ON=1
        img = ax.imshow(mat2.values, aspect="auto", interpolation="nearest",
                        vmin=0, vmax=1, cmap=cmap)
        
        # --- add horizontal white separators between cells (y only) ---
        n_rows = mat2.shape[0]
        ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)   # Row boundaries
        ax.grid(which="minor", axis="y", color="white", linewidth=0.3)
        ax.tick_params(which="minor", left=False, right=False)  # Don't show minor ticks
        
        #img = ax.imshow(mat2.values, aspect="auto", interpolation="nearest", vmin=0, vmax=1)
        ax.set_ylabel(nice_group(g))
        #ax.set_yticks([])

    axes[-1].set_xlabel("Time (min)")
    # tick every 10 min
    if max_frame > 0:
        step = int(10/frame_to_min) if frame_to_min > 0 else 10
        xticks = np.arange(0, max_frame, step=max(1, step))
        axes[-1].set_xticks(xticks)
        axes[-1].set_xticklabels([str(int(x*frame_to_min)) for x in xticks])

    # if title is None:
    #     title = f"{gene}: ON/OFF state raster (top {n_cells} cells / group)"
    fig.suptitle(title, y=1.02)

    if outname is None:
        outname = f"{gene}_state_raster_groups{'-'.join(groups)}.pdf"
    savefig_pdf(fig, FIGDIR / outname)

In [ ]:
# --- Fig. 5 main candidates: raster, duty cycle ---

for ds in datasets:
    gene = ds["gene"]
    runs = ds["runs"]
    perframe = ds["perframe"]

    print("\n=== ", gene, "===")

    # (1) state raster
    plot_state_raster(perframe, gene=gene, groups=GROUP_ORDER_MAIN, n_cells=100,
                      outname=f"Fig5_{gene}_state_raster_MAIN.pdf")

    # (2) duty cycle
    dc = plot_duty_cycle(perframe, gene=gene, groups=GROUP_ORDER_MAIN,
                         outname=f"Fig5_{gene}_duty_cycle_MAIN.pdf")
    dc.pipe(export_transcription_states).to_csv(TABLEDIR / f"{gene}_duty_cycle_celllevel_MAIN.csv", index=False)

This section produces more detailed plots and QC summaries that are typically better suited for **Extended Data**:

- Full Kaplan–Meier curves for **Active** and **Inactive** durations (all 4 groups).
- QC table: number of cells, number of runs, censoring fractions, mean/quantiles.

(Parametric model fits / mixture models can also be placed here if needed, but KM/RMST are usually sufficient for a clean story.)


In [ ]:
# --- ED: QC table + full KM curves (ON/OFF) ---

def qc_table(runs: pd.DataFrame, groups) -> pd.DataFrame:
    df = runs[runs["group"].isin(groups)].copy()

    qc_cells = (df[["group","cell_uid"]]
                .drop_duplicates()
                .groupby("group")["cell_uid"].nunique()
                .rename("n_cells")
                .reset_index())

    qc_runs = (df.groupby(["group","kind"])
               .agg(
                   n_runs=("length_min","size"),
                   frac_left=("left_censored","mean"),
                   frac_right=("right_censored","mean"),
                   mean_len=("length_min","mean"),
                   p90_len=("length_min", lambda x: np.percentile(x, 90)),
               )
               .reset_index())

    out = qc_runs.merge(qc_cells, on="group", how="left")
    return out.sort_values(["kind","group"])


def max_event_censor_table(runs: pd.DataFrame, groups) -> pd.DataFrame:
    """[R3-M4] Per (group, kind): maximum EVENT time (observed run, i.e. NOT right-censored)
    and maximum CENSOR time (right-censored run), AFTER excluding left-censored runs.
    Supports (a) the Methods sentence on right-censoring / the 60-min dwell bound and
    (b) the figure source-data files that the reviewer asked to confirm."""
    df = runs[runs["group"].isin(groups)].copy()
    df = df[~df["left_censored"].astype(bool)].copy()            # 'after left-censored run handling'
    df = df[np.isfinite(df["length_min"]) & (df["length_min"] > 0)]
    rows = []
    for (g, k), d in df.groupby(["group", "kind"]):
        ev = d[~d["right_censored"].astype(bool)]["length_min"]
        ce = d[ d["right_censored"].astype(bool)]["length_min"]
        rows.append(dict(
            group=g, kind=k,
            n_event=int(len(ev)),  max_event_min=(float(ev.max()) if len(ev) else np.nan),
            n_censor=int(len(ce)), max_censor_min=(float(ce.max()) if len(ce) else np.nan),
        ))
    return pd.DataFrame(rows).sort_values(["kind", "group"])

for ds in datasets:
    gene = ds["gene"]
    runs = ds["runs"]
    # [R3-M4] KM x-axis upper bound = true movie duration (robust to 60 vs 61 frames)
    _km_xmax = float(ds["perframe"].groupby("cell_uid")["frame"].nunique().max()) * (DELTA_MIN if "DELTA_MIN" in globals() else 1.0)

    qc = qc_table(runs, groups=GROUP_ORDER_FULL)
    display(qc)
    qc.pipe(export_transcription_states).to_csv(TABLEDIR / f"{gene}_QC_runs_cells_FULL.csv", index=False)

    # [R3-M4] max event / censor times per group & kind (source-data + Methods)
    mec = max_event_censor_table(runs, groups=GROUP_ORDER_FULL)
    display(mec)
    mec.pipe(export_transcription_states).to_csv(TABLEDIR / f"MaxEventCensorTimes_{gene}.csv", index=False)

    # Full KM (exclude left-censored; include right-censored)
    # [R3-M4] Display on a UNIFORM 0-T_max axis (= true movie duration; robust to 60 vs 61 frames).
    plot_km_panel(runs, kind="ON",  gene=gene, groups=GROUP_ORDER_FULL, exclude_left=True, logy=True,
                  xlim=(0, _km_xmax),
                  outname=f"ED_{gene}_KM_ON_FULL.pdf", title=f"$\\it{{{gene}}}$: KM active (exclude left-censored)")
    plot_km_panel(runs, kind="OFF", gene=gene, groups=GROUP_ORDER_FULL, exclude_left=True, logy=True,
                  xlim=(0, _km_xmax),
                  outname=f"ED_{gene}_KM_OFF_FULL.pdf", title=f"$\\it{{{gene}}}$: KM inactive (exclude left-censored)")

In [ ]:
# ============================================================
# Threshold sensitivity QC  (reviewer R3-m11)  --  PLACE THIS CELL LAST
# ------------------------------------------------------------
# Purpose: show the main conclusions (TSA / RGFP966 effect vs DMSO) do
#          NOT depend on the relaxed (rescue) center-determination
#          criterion, by comparing the two Cell-7 runs side by side.
#
# HOW THE INPUTS ARE PRODUCED:
#   The first code cell now runs the pipeline under BOTH settings automatically:
#   USE_TETR_RESCUE = True   -> outputs .../tetr_mcp_hybrid_analysis           (PAPER)
#   USE_TETR_RESCUE = False  -> outputs .../tetr_mcp_hybrid_analysis_standard_only
#   No manual re-run needed; this cell reads both folders.
#
# Confirmed parameters (for Methods): mTetR standard threshold = 1.10,
#   MCP threshold = 1.25, rescue threshold = 0.935 (= 1.10 x 0.85),
#   rescue window = 6 px, nearest-neighbor linking = 3 px,
#   min consecutive frames = 3.
#
# Metrics per (gene, config, group): n_cells, n_runs, Active fraction
#   (= duty cycle), median Active/Inactive dwell, mean MCP intensity.
# Stats  per (gene, config): TSA/RGFP966 vs DMSO for Active fraction
#   (Mann-Whitney + Holm) and for dwell-time KM (log-rank).
# Verdict: is the direction + significance of each effect preserved
#   across the two configs?  ("YES" = conclusion not threshold-dependent)
#
# Reuses: datasets, GROUP_ORDER_MAIN, _ensure_cell_uid, TABLEDIR,
#   mannwhitneyu, holm_adjust_array, display.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import chi2

CONFIGS   = {"rescue": "", "standard_only": "_standard_only"}   # name -> dir suffix
QC_GROUPS = GROUP_ORDER_MAIN
QC_BASE, QC_COMPS = "DMSO", ["TSA", "RGFP966"]
SIG = 0.05


def _load_config_dir(base_dir, suffix):
    """Load perframe + runs for one config; returns (perframe, runs, dir)."""
    d = Path(base_dir).parent / (Path(base_dir).name + suffix)
    pf_p = d / "aggregated_perframe.csv"
    rb_p = d / "aggregated_runs_by_cell.csv"
    rr_p = d / "aggregated_runs.csv"
    if not pf_p.exists():
        return None, None, d
    perframe = _ensure_cell_uid(read_legacy_state_csv(pf_p))
    if rb_p.exists():
        runs = _ensure_cell_uid(read_legacy_state_csv(rb_p))
    elif rr_p.exists():
        runs = _ensure_cell_uid(read_legacy_state_csv(rr_p))
    else:
        runs = None
    return perframe, runs, d


def _prep_runs(runs, kind, groups):
    """Exclude left-censored; event = not right-censored (same convention as KM)."""
    df = runs[(runs["kind"] == kind) & (runs["group"].isin(groups))].copy()
    df = df[~df["left_censored"]].copy()
    df["event"] = (~df["right_censored"]).astype(int)
    df = df[np.isfinite(df["length_min"]) & (df["length_min"] > 0)].copy()
    return df


def _logrank_p(t1, e1, t2, e2):
    """Two-group log-rank test -> p-value (chi-square, df = 1)."""
    t1 = np.asarray(t1, float); e1 = np.asarray(e1, int)
    t2 = np.asarray(t2, float); e2 = np.asarray(e2, int)
    if len(t1) < 2 or len(t2) < 2:
        return np.nan
    times = np.unique(np.concatenate([t1[e1 == 1], t2[e2 == 1]]))
    O1 = E1 = V = 0.0
    for t in times:
        n1 = np.sum(t1 >= t); n2 = np.sum(t2 >= t); n = n1 + n2
        d1 = np.sum((t1 == t) & (e1 == 1)); d = d1 + np.sum((t2 == t) & (e2 == 1))
        if n < 2 or d == 0:
            continue
        O1 += d1
        E1 += d * n1 / n
        V  += d * (n1 / n) * (n2 / n) * (n - d) / (n - 1)
    if V <= 0:
        return np.nan
    return float(chi2.sf((O1 - E1) ** 2 / V, 1))


# ---- collect descriptive metrics + per-config stats ----
desc_rows, stat_rows = [], []
for ds in datasets:
    gene, base_dir = ds["gene"], ds["dir"]
    for cname, suffix in CONFIGS.items():
        perframe, runs, d = _load_config_dir(base_dir, suffix)
        if perframe is None:
            print(f"[SKIP] {gene}/{cname}: not found -> {d}")
            continue

        # per-cell Active fraction (= duty cycle)
        dc = (perframe[perframe.group.isin(QC_GROUPS) & np.isfinite(perframe.on_off)]
              .groupby(["group", "cell_uid"])["on_off"].mean()
              .reset_index().rename(columns={"on_off": "active_fraction"}))
        has_int = "mcp_integrated_intensity" in perframe.columns

        # descriptive metrics per group
        for g in QC_GROUPS:
            cells = dc[dc.group == g]["active_fraction"]
            on  = runs[(runs.group == g) & (runs.kind == "ON")]["length_min"]  if runs is not None else pd.Series(dtype=float)
            off = runs[(runs.group == g) & (runs.kind == "OFF")]["length_min"] if runs is not None else pd.Series(dtype=float)
            intens = (perframe[(perframe.group == g) & (perframe.on_off == 1)]["mcp_integrated_intensity"].mean()
                      if has_int else np.nan)
            desc_rows.append(dict(
                gene=gene, config=cname, group=g,
                n_cells=int(cells.notna().sum()),
                n_runs_ON=int(len(on)), n_runs_OFF=int(len(off)),
                active_fraction_mean=float(cells.mean()) if len(cells) else np.nan,
                median_ON_min=float(on.median()) if len(on) else np.nan,
                median_OFF_min=float(off.median()) if len(off) else np.nan,
                mcp_intensity_active_mean=float(intens) if np.isfinite(intens) else np.nan,
            ))

        # stats: Active fraction, comp vs DMSO (Mann-Whitney + Holm)
        base_af = dc[dc.group == QC_BASE]["active_fraction"].dropna().values
        raw, comps = [], []
        for g in QC_COMPS:
            v = dc[dc.group == g]["active_fraction"].dropna().values
            p = mannwhitneyu(base_af, v, alternative="two-sided").pvalue if (len(base_af) >= 2 and len(v) >= 2) else np.nan
            raw.append(p); comps.append(g)
        adj = holm_adjust_array([0.0 if not np.isfinite(p) else p for p in raw])
        for g, p, pa in zip(comps, raw, adj):
            md = (dc[dc.group == g]["active_fraction"].mean() - base_af.mean()) if len(base_af) else np.nan
            stat_rows.append(dict(gene=gene, config=cname, metric="active_fraction",
                                  compare=f"{QC_BASE} vs {g}",
                                  effect_sign=("+" if md > 0 else "-") if np.isfinite(md) else "na",
                                  p=float(pa) if np.isfinite(p) else np.nan))

        # stats: dwell-time KM, comp vs DMSO (log-rank), ON and OFF
        if runs is not None:
            for kind in ["ON", "OFF"]:
                rk = _prep_runs(runs, kind, QC_GROUPS)
                b = rk[rk.group == QC_BASE]
                for g in QC_COMPS:
                    s = rk[rk.group == g]
                    p = _logrank_p(b["length_min"], b["event"], s["length_min"], s["event"])
                    md = (s["length_min"].median() - b["length_min"].median()) if (len(s) and len(b)) else np.nan
                    stat_rows.append(dict(gene=gene, config=cname, metric=f"KM_{kind}",
                                          compare=f"{QC_BASE} vs {g}",
                                          effect_sign=("+" if md > 0 else "-") if np.isfinite(md) else "na",
                                          p=float(p) if np.isfinite(p) else np.nan))

qc_desc  = pd.DataFrame(desc_rows)
qc_stats = pd.DataFrame(stat_rows)

# ---- verdict: sign + significance preserved across configs? ----
verdict_rows = []
if not qc_stats.empty and qc_stats["config"].nunique() >= 2:
    for (gene, metric, comp), grp in qc_stats.groupby(["gene", "metric", "compare"]):
        r = grp[grp.config == "rescue"]
        s = grp[grp.config == "standard_only"]
        if r.empty or s.empty:
            continue
        r, s = r.iloc[0], s.iloc[0]
        same_sign = (r.effect_sign == s.effect_sign) and (r.effect_sign != "na")
        same_sig  = np.isfinite(r.p) and np.isfinite(s.p) and ((r.p < SIG) == (s.p < SIG))
        verdict_rows.append(dict(gene=gene, metric=metric, compare=comp,
                                 rescue_sign=r.effect_sign, rescue_p=r.p,
                                 standard_sign=s.effect_sign, standard_p=s.p,
                                 consistent=("YES" if (same_sign and same_sig) else "CHECK")))
qc_verdict = pd.DataFrame(verdict_rows)

# ---- save + show ----
qc_desc.pipe(export_transcription_states).to_csv(TABLEDIR / "ThresholdSensitivity_descriptive.csv", index=False)
qc_stats.pipe(export_transcription_states).to_csv(TABLEDIR / "ThresholdSensitivity_stats.csv", index=False)
if not qc_verdict.empty:
    qc_verdict.pipe(export_transcription_states).to_csv(TABLEDIR / "ThresholdSensitivity_verdict.csv", index=False)

print("=== Descriptive metrics (standard vs rescue) ==="); display(qc_desc)
print("\n=== Effect vs DMSO, per config ==="); display(qc_stats)
if not qc_verdict.empty:
    print("\n=== Verdict: conclusions preserved across thresholds? ==="); display(qc_verdict)
else:
    print("\n[note] Run Cell 7 with BOTH USE_TETR_RESCUE=True and =False, "
          "then rerun this cell to get the cross-config verdict.")

In [ ]:
# ============================================================
# Threshold sensitivity QC -- visualization add-on
# ------------------------------------------------------------
# Append this to the END of the threshold-sensitivity QC cell
# (it reuses qc_desc computed above; it does NOT recompute anything).
#
# Produces, per gene, grouped bar charts comparing the two configs
# (rescue vs standard_only) side by side:
#   (1) Active fraction (= duty cycle)   -> the main-conclusion metric
#   (2) median Active / Inactive dwell   -> y fixed to (0, 60) min
# If only one config is present, it still plots that one (no crash).
#
# Reuses: qc_desc, GROUP_ORDER_MAIN, nice_group, FIGDIR, savefig_pdf.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# fixed colors so the two configs are visually consistent across panels
_CFG_COLOR = {"rescue": "#4C72B0", "standard_only": "#DD8452"}
_CFG_LABEL = {"rescue": "rescue (paper)", "standard_only": "standard only"}


def _grouped_bars(ax, sub, value_col, ylabel, title, ylim=None):
    """Grouped bar chart: x = groups, bars = configs present in `sub`."""
    configs = [c for c in ["rescue", "standard_only"] if c in sub["config"].unique()]
    groups = [g for g in GROUP_ORDER_MAIN if g in sub["group"].unique()]
    x = np.arange(len(groups))
    n = max(len(configs), 1)
    width = 0.8 / n

    for k, cfg in enumerate(configs):
        vals = []
        for g in groups:
            row = sub[(sub.config == cfg) & (sub.group == g)][value_col]
            vals.append(float(row.iloc[0]) if len(row) else np.nan)
        offset = (k - (n - 1) / 2.0) * width
        ax.bar(x + offset, vals, width=width,
               color=_CFG_COLOR.get(cfg, None), label=_CFG_LABEL.get(cfg, cfg))

    ax.set_xticks(x)
    ax.set_xticklabels([nice_group(g) for g in groups])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(frameon=False, fontsize=7)


# ---- one figure per gene ----
if "qc_desc" in globals() and not qc_desc.empty:
    for gene in qc_desc["gene"].unique():
        g_desc = qc_desc[qc_desc.gene == gene]

        # Panel 1: Active fraction (main conclusion metric)
        fig, ax = plt.subplots(figsize=(3.4, 2.6))
        _grouped_bars(ax, g_desc, "active_fraction_mean",
                      "Active fraction (mean per cell)",
                      f"$\\it{{{gene}}}$: Active fraction (threshold check)",
                      ylim=(0, 1.0))
        savefig_pdf(fig, FIGDIR / f"ThresholdSensitivity_{gene}_active_fraction.pdf")

        # Panel 2: median dwell times (ON and OFF), y fixed to movie duration
        fig, axes = plt.subplots(1, 2, figsize=(5.4, 2.6))
        _grouped_bars(axes[0], g_desc, "median_ON_min",
                      "Median Active dwell (min)",
                      f"$\\it{{{gene}}}$: Active dwell", ylim=(0, 60))
        _grouped_bars(axes[1], g_desc, "median_OFF_min",
                      "Median Inactive dwell (min)",
                      f"$\\it{{{gene}}}$: Inactive dwell", ylim=(0, 60))
        fig.tight_layout()
        savefig_pdf(fig, FIGDIR / f"ThresholdSensitivity_{gene}_dwell.pdf")
else:
    print("[note] qc_desc is empty - run the QC table part above first.")

In [ ]:
# ============================================================
# Per-window time course  (user-added QC; reviewer R3-M5 / R3-M6)
# ------------------------------------------------------------
# Goal : with NO assumed split point, scan the whole movie in equal
#        windows to see WHEN the Active fraction changes vs plateaus.
#        Use this to pick a data-driven early/late boundary.
# Metric: per-cell Active fraction averaged within each fixed window,
#         shown as population mean +/- SEM over time.
# Plot  : Active fraction over time -> line  (no dwell plots, per request).
#
# Reuses from earlier cells: datasets, GROUP_ORDER_MAIN, nice_group,
#   _ensure_cell_uid, FIGDIR, TABLEDIR, savefig_pdf, display.
#
# NOTE: t0 = imaging onset is ASSUMED (verify drug timing).
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Config (edit here) ----
FRAME_MIN_TC  = DELTA_MIN if "DELTA_MIN" in globals() else 1.0   # minutes per frame
T0_MIN_TC     = 0.0          # drug-addition time (min); verify, then edit
WINDOW_WIDTH  = 5.0          # <-- change ONLY this number for a different width
MOVIE_DUR     = 60.0         # total movie length (min)
MIN_FRAMES_TC = 3            # min frames in a window to score a cell
TC_GROUPS     = GROUP_ORDER_MAIN

# equal, non-overlapping windows: [0,w), [w,2w), ... , [.., MOVIE_DUR]
_edges = np.arange(0.0, MOVIE_DUR + 1e-9, WINDOW_WIDTH)
TC_WINDOWS = [(float(_edges[i]), float(_edges[i + 1])) for i in range(len(_edges) - 1)]


def _window_active_fraction(perframe):
    """Per-cell Active fraction in each fixed window -> long df."""
    pf = _ensure_cell_uid(perframe).copy()
    pf = pf[pf["group"].isin(TC_GROUPS) & np.isfinite(pf["on_off"])].copy()
    pf["time_min"] = pf["frame"].astype(float) * FRAME_MIN_TC - T0_MIN_TC
    n_win = len(TC_WINDOWS)
    rows = []
    for (g, cuid), d in pf.groupby(["group", "cell_uid"], sort=False):
        tv = d["time_min"].to_numpy()
        for wi, (lo, hi) in enumerate(TC_WINDOWS):
            last = (wi == n_win - 1)
            mask = (tv >= lo) & (tv <= hi + 1e-9) if last else (tv >= lo) & (tv < hi)
            sub = d[mask]
            af = float(sub["on_off"].mean()) if len(sub) >= MIN_FRAMES_TC else np.nan
            rows.append(dict(group=g, cell_uid=cuid, win_idx=wi,
                             win_mid=(lo + hi) / 2.0, active_fraction=af))
    return pd.DataFrame(rows)


def _summarize(af_long):
    """Population mean +/- SEM per (group, window)."""
    g = af_long.groupby(["group", "win_idx", "win_mid"])["active_fraction"]
    return (g.mean().rename("mean").reset_index()
              .merge(g.sem().rename("sem").reset_index(),
                     on=["group", "win_idx", "win_mid"])
              .sort_values(["group", "win_idx"]))


def _plot_timecourse(summary, gene):
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    for g in TC_GROUPS:
        s = summary[summary.group == g].sort_values("win_mid")
        if not s["mean"].isna().all():
            ax.errorbar(s["win_mid"], s["mean"], yerr=s["sem"], marker="o", ms=4,
                        lw=1.2, capsize=2, label=nice_group(g))
    ax.set_xlim(0, MOVIE_DUR); ax.set_ylim(0, 1.0)
    ax.set_xlabel(f"Time after imaging onset (min), {int(WINDOW_WIDTH)}-min windows\n"
                  f"(t0 = imaging onset; verify drug timing)")
    ax.set_ylabel("Active fraction")
    ax.set_title(f"$\\it{{{gene}}}$: Active fraction over time")
    ax.legend(frameon=False); ax.grid(alpha=0.3)
    savefig_pdf(fig, FIGDIR / f"TimeCourse_{gene}_w{int(WINDOW_WIDTH)}_active_fraction.pdf")


# ---- Run ----
print(f"[time course] window={WINDOW_WIDTH} min -> {len(TC_WINDOWS)} windows, "
      f"frame={FRAME_MIN_TC} min")

for ds in datasets:
    gene, perframe = ds["gene"], ds["perframe"]
    af_long = _window_active_fraction(perframe)
    summary = _summarize(af_long)

    af_long.pipe(export_transcription_states).to_csv(TABLEDIR / f"TimeCourse_{gene}_w{int(WINDOW_WIDTH)}_active_fraction_cell.csv", index=False)
    summary.pipe(export_transcription_states).to_csv(TABLEDIR / f"TimeCourse_{gene}_w{int(WINDOW_WIDTH)}_active_fraction_summary.csv", index=False)
    _plot_timecourse(summary, gene)
    print(f"\n=== Time course ({int(WINDOW_WIDTH)}-min): {gene} ==="); display(summary)

In [ ]:
# ============================================================
# Early-vs-late window analysis  (reviewer R3-M6)
# ------------------------------------------------------------
# Goal : test whether the HDAC-inhibitor effect on transcription
#        is driven by an early post-treatment transition or
#        persists in a later (steady) window.
# Metric: per-cell Active fraction (= duty cycle = mean of on_off).
# Plots : (1) population Active-fraction time course      -> line
#         (2) per-cell Active fraction, early vs late      -> boxplot
# (No KM curves, per request.)
#
# Reuses from earlier cells: datasets, GROUP_ORDER_MAIN, nice_group,
#   _ensure_cell_uid, FIGDIR, TABLEDIR, savefig_pdf, mannwhitneyu,
#   holm_adjust_array, display.
#
# NOTE: drug-addition time is NOT recorded; t0 = imaging onset is ASSUMED.
#       Confirm the real timing and edit T0_MIN if needed.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Config (edit here) ----
FRAME_MIN  = DELTA_MIN if "DELTA_MIN" in globals() else 1.0   # minutes per frame
T0_MIN     = 0.0            # drug-addition time (min); verify, then edit
EARLY_WIN  = (0.0, 15.0)    # [start, end)  min, relative to t0
LATE_WIN   = (15.0, 60.0)   # [start, end]  min, relative to t0
MIN_FRAMES = 3              # min frames in a window to score a cell
EL_GROUPS  = GROUP_ORDER_MAIN
BASE, COMPS = "DMSO", ["TSA", "RGFP966"]


def _per_cell_active_fraction(perframe):
    """Per-cell Active fraction in the early and late windows -> long df."""
    pf = _ensure_cell_uid(perframe).copy()
    pf = pf[pf["group"].isin(EL_GROUPS) & np.isfinite(pf["on_off"])].copy()
    pf["time_min"] = pf["frame"].astype(float) * FRAME_MIN - T0_MIN
    rows = []
    for (g, cuid), d in pf.groupby(["group", "cell_uid"], sort=False):
        tv = d["time_min"].to_numpy()
        for wname, (lo, hi), is_last in [("early", EARLY_WIN, False),
                                         ("late",  LATE_WIN,  True)]:
            mask = (tv >= lo) & (tv <= hi + 1e-9) if is_last else (tv >= lo) & (tv < hi)
            sub = d[mask]
            af = float(sub["on_off"].mean()) if len(sub) >= MIN_FRAMES else np.nan
            rows.append(dict(group=g, cell_uid=cuid, window=wname, active_fraction=af))
    return pd.DataFrame(rows)


def _population_timecourse(perframe, bin_min=2.0):
    """Population mean Active fraction in time bins -> long df."""
    pf = _ensure_cell_uid(perframe).copy()
    pf = pf[pf["group"].isin(EL_GROUPS) & np.isfinite(pf["on_off"])].copy()
    pf["time_min"] = pf["frame"].astype(float) * FRAME_MIN - T0_MIN
    edges = np.arange(0.0, 60.0 + bin_min, bin_min)
    pf["t_mid"] = pd.cut(pf["time_min"], bins=edges, right=False,
                         labels=(edges[:-1] + bin_min / 2.0))
    tc = (pf.groupby(["group", "t_mid"], observed=True)["on_off"].mean()
            .reset_index().rename(columns={"on_off": "active_fraction"}))
    tc["t_mid"] = tc["t_mid"].astype(float)
    return tc


def _compare_active_fraction(af_long, gene):
    """MW + Holm: comp vs DMSO in each window; plus a late-early (DiD) test."""
    rows = []
    for wname in ["early", "late"]:
        base = af_long[(af_long.window == wname) & (af_long.group == BASE)]["active_fraction"].dropna().values
        raw, comps = [], []
        for g in COMPS:
            v = af_long[(af_long.window == wname) & (af_long.group == g)]["active_fraction"].dropna().values
            p = mannwhitneyu(base, v, alternative="two-sided").pvalue if (len(base) >= 2 and len(v) >= 2) else np.nan
            raw.append(p); comps.append(g)
        adj = holm_adjust_array([0.0 if not np.isfinite(p) else p for p in raw])
        for g, p, pa in zip(comps, raw, adj):
            rows.append(dict(gene=gene, window=wname, compare=f"{BASE} vs {g}",
                             p_raw=p, p_holm=(pa if np.isfinite(p) else np.nan)))
    # difference-in-differences: per-cell (late - early), comp vs DMSO
    wide = af_long.pivot_table(index=["group", "cell_uid"], columns="window",
                               values="active_fraction").reset_index()
    if {"early", "late"}.issubset(wide.columns):
        wide["delta"] = wide["late"] - wide["early"]
        base_d = wide[wide.group == BASE]["delta"].dropna().values
        raw, comps = [], []
        for g in COMPS:
            v = wide[wide.group == g]["delta"].dropna().values
            p = mannwhitneyu(base_d, v, alternative="two-sided").pvalue if (len(base_d) >= 2 and len(v) >= 2) else np.nan
            raw.append(p); comps.append(g)
        adj = holm_adjust_array([0.0 if not np.isfinite(p) else p for p in raw])
        for g, p, pa in zip(comps, raw, adj):
            rows.append(dict(gene=gene, window="late-early (DiD)", compare=f"{BASE} vs {g}",
                             p_raw=p, p_holm=(pa if np.isfinite(p) else np.nan)))
    return pd.DataFrame(rows)


def _plot_timecourse(tc, gene):
    fig, ax = plt.subplots(figsize=(4.0, 2.8))
    for g in EL_GROUPS:
        s = tc[tc.group == g].sort_values("t_mid")
        if not s.empty:
            ax.plot(s["t_mid"], s["active_fraction"], marker="o", ms=3, lw=1.2, label=nice_group(g))
    ax.axvspan(EARLY_WIN[0], EARLY_WIN[1], color="0.9", lw=0)   # shade early window
    ax.axvline(EARLY_WIN[1], color="0.5", ls="--", lw=0.8)      # early/late boundary
    ax.set_xlim(0, 60); ax.set_ylim(0, 1.0)
    ax.set_xlabel("Time after imaging onset (min)\n(t0 = imaging onset; verify drug timing)")
    ax.set_ylabel("Active fraction (population)")
    ax.set_title(f"$\\it{{{gene}}}$: active fraction over time")
    ax.legend(frameon=False); ax.grid(alpha=0.3)
    savefig_pdf(fig, FIGDIR / f"EarlyLate_{gene}_active_fraction_timecourse.pdf")


def _plot_early_late_box(af_long, gene):
    fig, ax = plt.subplots(figsize=(3.8, 2.8))
    rng = np.random.default_rng(0)
    xticks, xlabels = [], []
    for i, g in enumerate(EL_GROUPS):
        for j, wname in enumerate(["early", "late"]):
            pos = i * 3 + j                       # 2 boxes per group, gap between groups
            y = af_long[(af_long.group == g) & (af_long.window == wname)]["active_fraction"].dropna().values
            ax.boxplot([y], positions=[pos], widths=0.6, showfliers=False)
            ax.plot(rng.normal(pos, 0.06, size=len(y)), y, "o", ms=3, alpha=0.6)
            xticks.append(pos); xlabels.append(wname)
        ax.text(i * 3 + 0.5, -0.16, nice_group(g), ha="center", va="top",
                transform=ax.get_xaxis_transform(), fontsize=8)
    ax.set_xticks(xticks); ax.set_xticklabels(xlabels)
    ax.set_ylim(0, 1.0); ax.set_ylabel("Active fraction (per cell)")
    ax.set_title(f"$\\it{{{gene}}}$: early vs late active fraction")
    ax.grid(axis="y", alpha=0.3)
    savefig_pdf(fig, FIGDIR / f"EarlyLate_{gene}_duty_by_window.pdf")


# ---- Run ----
print(f"[Early-vs-late] t0={T0_MIN} min (assumed = imaging onset), "
      f"early={EARLY_WIN}, late={LATE_WIN}, frame={FRAME_MIN} min")

el_stats_all = []
for ds in datasets:
    gene, perframe = ds["gene"], ds["perframe"]
    af_long = _per_cell_active_fraction(perframe)
    tc      = _population_timecourse(perframe)
    stats   = _compare_active_fraction(af_long, gene)
    el_stats_all.append(stats)

    af_long.pipe(export_transcription_states).to_csv(TABLEDIR / f"EarlyLate_{gene}_active_fraction.csv", index=False)
    _plot_timecourse(tc, gene)
    _plot_early_late_box(af_long, gene)
    print(f"\n=== Early-vs-late: {gene} ==="); display(stats)

if el_stats_all:
    el_stats = pd.concat(el_stats_all, ignore_index=True)
    el_stats.pipe(export_transcription_states).to_csv(TABLEDIR / "EarlyLate_stats_summary.csv", index=False)
    print("\n[saved] EarlyLate_stats_summary.csv"); display(el_stats)

In [ ]:
# ============================================================
# Time-segmented Active-fraction panel  (reviewer R3-M6, extra figure)
# ------------------------------------------------------------
# Same metric + style as the main Fig.5 duty-cycle panel, but split into
# fixed, equal time windows so the reviewer can see whether the TSA /
# RGFP966 effect is an early transient or is sustained across the movie.
#
#   x-axis : time windows (default 0-20, 20-40, 40-60 min)
#   within each window: DMSO / TSA / RGFP966  (boxplot + jittered points)
#   stats  : per window, TSA vs DMSO and RGFP966 vs DMSO (Mann-Whitney + Holm)
#
# This is the "another justified split" permitted by the R3-M6 action item;
# the 0-15 / 15-60 early-vs-late panel (previous cell) is KEPT as well, so
# the literal action item and this cleaner trajectory view are both covered.
#
# Reuses from earlier cells: datasets, GROUP_ORDER_MAIN, nice_group,
#   _ensure_cell_uid, FIGDIR, TABLEDIR, savefig_pdf, mannwhitneyu,
#   holm_adjust_array, p_to_stars, add_sig_bracket, display.
#
# NOTE: t0 = imaging onset is ASSUMED (drug-addition timing not recorded);
#       edit SEG_T0_MIN once the real timing is confirmed with the boss.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Config (edit here) ----
SEG_WINDOWS    = [(0.0, 20.0), (20.0, 40.0), (40.0, 60.0)]  # equal 20-min windows; change here for another split
SEG_FRAME_MIN  = DELTA_MIN if "DELTA_MIN" in globals() else 1.0
SEG_T0_MIN     = 0.0        # drug-addition time (min); verify, then edit
SEG_MIN_FRAMES = 5          # min frames within a window to score a cell (20-frame window -> >=5 == >=25%)
SEG_GROUPS     = GROUP_ORDER_MAIN
SEG_BASE, SEG_COMPS = "DMSO", ["TSA", "RGFP966"]

# group colors matched to the main duty-cycle panel (matplotlib C0/C1/C2)
SEG_GROUP_COLOR = {"DMSO": "C0", "TSA": "C1", "RGFP966": "C2"}


def _seg_active_fraction(perframe):
    """Per-cell Active fraction inside each SEG_WINDOWS window -> long df."""
    pf = _ensure_cell_uid(perframe).copy()
    pf = pf[pf["group"].isin(SEG_GROUPS) & np.isfinite(pf["on_off"])].copy()
    pf["time_min"] = pf["frame"].astype(float) * SEG_FRAME_MIN - SEG_T0_MIN
    n_win = len(SEG_WINDOWS)
    rows = []
    for (g, cuid), d in pf.groupby(["group", "cell_uid"], sort=False):
        tv = d["time_min"].to_numpy()
        for wi, (lo, hi) in enumerate(SEG_WINDOWS):
            last = (wi == n_win - 1)
            mask = (tv >= lo) & (tv <= hi + 1e-9) if last else (tv >= lo) & (tv < hi)
            sub = d[mask]
            af = float(sub["on_off"].mean()) if len(sub) >= SEG_MIN_FRAMES else np.nan
            rows.append(dict(group=g, cell_uid=cuid, win_idx=wi,
                             win_label=f"{int(lo)}-{int(hi)} min", active_fraction=af))
    return pd.DataFrame(rows)


def _seg_pvals(af_long):
    """Per window: TSA/RGFP966 vs DMSO (Mann-Whitney + Holm WITHIN each window)."""
    rows = []
    for wi, (lo, hi) in enumerate(SEG_WINDOWS):
        wl = f"{int(lo)}-{int(hi)} min"
        base = af_long[(af_long.win_idx == wi) & (af_long.group == SEG_BASE)]["active_fraction"].dropna().values
        raw, comps = [], []
        for g in SEG_COMPS:
            v = af_long[(af_long.win_idx == wi) & (af_long.group == g)]["active_fraction"].dropna().values
            p = mannwhitneyu(base, v, alternative="two-sided").pvalue if (len(base) >= 2 and len(v) >= 2) else np.nan
            raw.append(p); comps.append(g)
        adj = holm_adjust_array([0.0 if not np.isfinite(p) else p for p in raw])
        for g, p, pa in zip(comps, raw, adj):
            rows.append(dict(win_idx=wi, window=wl, compare=f"{SEG_BASE} vs {g}",
                             p_raw=(float(p) if np.isfinite(p) else np.nan),
                             p_holm=(float(pa) if np.isfinite(p) else np.nan)))
    return pd.DataFrame(rows)


def _plot_seg_box(af_long, pvals, gene):
    """Grouped boxplot + jittered points, styled like the Fig.5 duty-cycle panel."""
    n_g = len(SEG_GROUPS)
    slot = n_g + 1                       # groups per window + 1-slot gap between windows
    fig, ax = plt.subplots(figsize=(1.15 * len(SEG_WINDOWS) * n_g, 2.6))
    jit = np.random.default_rng(0)

    pos_of = {}                          # (win_idx, group) -> x position
    for wi, (lo, hi) in enumerate(SEG_WINDOWS):
        for gi, g in enumerate(SEG_GROUPS):
            pos = wi * slot + gi
            pos_of[(wi, g)] = pos
            y = af_long[(af_long.win_idx == wi) & (af_long.group == g)]["active_fraction"].dropna().values
            ax.boxplot([y], positions=[pos], widths=0.6, showfliers=False)
            ax.plot(jit.normal(pos, 0.06, size=len(y)), y, "o", ms=3, alpha=0.6,
                    color=SEG_GROUP_COLOR.get(g, None))

    # within-window significance brackets (DMSO vs TSA, DMSO vs RGFP966)
    y_top = np.nanmax(af_long["active_fraction"].values)
    y0 = (y_top if np.isfinite(y_top) else 1.0) + 0.03
    for wi in range(len(SEG_WINDOWS)):
        sub = pvals[pvals.win_idx == wi]
        for k, (_, r) in enumerate(sub.iterrows()):
            g = r["compare"].split(" vs ")[1]
            x1 = pos_of[(wi, SEG_BASE)]; x2 = pos_of[(wi, g)]
            lab = p_to_stars(r["p_holm"]) if np.isfinite(r["p_holm"]) else "n/a"
            add_sig_bracket(ax, x1, x2, y0 + k * 0.12, lab, h=0.015, lw=0.8)

    # per-group ticks + a window label centered under each triple
    xticks, xlabels = [], []
    for wi, (lo, hi) in enumerate(SEG_WINDOWS):
        for g in SEG_GROUPS:
            xticks.append(pos_of[(wi, g)]); xlabels.append(nice_group(g))
        center = float(np.mean([pos_of[(wi, g)] for g in SEG_GROUPS]))
        ax.text(center, -0.22, f"{int(lo)}-{int(hi)} min", ha="center", va="top",
                transform=ax.get_xaxis_transform(), fontsize=8)
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels, rotation=30, ha="right", fontsize=7)
    ax.set_ylim(0, 1.0 + 0.12 * len(SEG_COMPS))
    ax.set_ylabel("Fraction of Active frames")
    ax.set_title(f"$\\it{{{gene}}}$: Active fraction by time window")
    ax.grid(axis="y", alpha=0.3)
    savefig_pdf(fig, FIGDIR / f"Fig5_{gene}_duty_cycle_by_window.pdf")


# ---- Run ----
print(f"[time-segmented] windows={SEG_WINDOWS}, t0={SEG_T0_MIN} min (assumed=imaging onset), "
      f"frame={SEG_FRAME_MIN} min, min_frames_per_window={SEG_MIN_FRAMES}")

seg_stats_all = []
for ds in datasets:
    gene, perframe = ds["gene"], ds["perframe"]
    af_long = _seg_active_fraction(perframe)
    pvals   = _seg_pvals(af_long)
    pvals.insert(0, "gene", gene)
    seg_stats_all.append(pvals)

    af_long.pipe(export_transcription_states).to_csv(TABLEDIR / f"Fig5_{gene}_duty_cycle_by_window_cell.csv", index=False)
    _plot_seg_box(af_long, pvals, gene)
    print(f"\n=== Time-segmented Active fraction: {gene} ==="); display(pvals)

if seg_stats_all:
    seg_stats = pd.concat(seg_stats_all, ignore_index=True)
    seg_stats.pipe(export_transcription_states).to_csv(TABLEDIR / "Fig5_duty_cycle_by_window_stats.csv", index=False)
    print("\n[saved] Fig5_duty_cycle_by_window_stats.csv"); display(seg_stats)


## R3-m11: mTetR and MCP threshold sensitivity

This section varies one detection threshold at a time. The other threshold is
held at the value used for the main analysis. The mTetR sweep uses rescue off
so that each condition has one unambiguous center-detection threshold. The MCP
sweep retains the paper mTetR setting and rescue rule.

Outputs are isolated under `threshold_sensitivity_outputs_20260811/threshold_sweeps/`:

- `runs/`: per-threshold per-frame and censoring-aware run tables
- `tables/`: cell-level values, run-level values, QC, and effect-size tables
- `figs/`: one compact supplementary PDF and a PNG preview

Dwell-time sensitivity is summarized by the restricted mean survival time
(RMST; tau = 30 min). Confidence intervals are obtained by resampling cells,
preserving all runs from each sampled cell and their censoring indicators.


In [ ]:
# R3-m11 configuration and local output folders
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

if "OUTPUTS_DIR" not in globals():
    raise RuntimeError("Run the main analysis-definition cell first.")

SENS_ROOT = Path(OUTPUTS_DIR) / "threshold_sweeps"
SENS_RUNS_DIR = SENS_ROOT / "runs"
SENS_TABLES_DIR = SENS_ROOT / "tables"
SENS_FIGS_DIR = SENS_ROOT / "figs"
for _path in (SENS_RUNS_DIR, SENS_TABLES_DIR, SENS_FIGS_DIR):
    _path.mkdir(parents=True, exist_ok=True)

THRESHOLD_SWEEPS = {
    "mtetr": {
        "values": [0.85, 1.00, 1.10, 1.30, 1.50],
        "paper_value": 1.10,
        "snapshot_value": 1.30,
        "fixed_mcp": 1.25,
        "use_tetr_rescue": False,
    },
    "mcp": {
        "values": [0.85, 1.00, 1.15, 1.25, 1.50],
        "paper_value": 1.25,
        "snapshot_value": 1.15,
        "fixed_tetr": 1.10,
        "use_tetr_rescue": True,
    },
}

SENS_GROUPS = ["DMSO", "TSA", "RGFP966"]
SENS_COMPARISONS = ["TSA", "RGFP966"]
MIN_CELL_FRAMES = 3
RMST_TAU_MIN = 30.0
BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_SEED = 20260811

# Existing complete threshold folders are reused. Set this to True only
# when the detection output itself must be regenerated.
OVERWRITE_EXISTING_SWEEPS = False

print(f"Sensitivity output root: {SENS_ROOT.resolve()}")
print(json.dumps(THRESHOLD_SWEEPS, indent=2))


In [ ]:
# Execute independent mTetR and MCP threshold sweeps.
def _threshold_run_dir(sweep_type: str, gene: str, threshold: float) -> Path:
    return (
        SENS_RUNS_DIR
        / sweep_type
        / f"threshold_{threshold:.3f}"
        / gene
    )


def _aggregate_threshold_perframe(outdir: Path) -> pd.DataFrame:
    per_file = sorted(
        path for path in outdir.glob("*_perframe.csv")
        if not path.name.startswith("aggregated")
    )
    if not per_file:
        raise FileNotFoundError(f"No per-frame CSV files were generated in {outdir}")
    perframe = pd.concat(
        [read_legacy_state_csv(path) for path in per_file], ignore_index=True
    )
    perframe = dedup_perframe(perframe, tag="[threshold sweep]")
    perframe = perframe.sort_values(
        ["group", "file", "cell_id", "frame"]
    ).reset_index(drop=True)
    perframe.pipe(export_transcription_states).to_csv(outdir / "aggregated_perframe.csv", index=False)
    return perframe


def _remove_generated_threshold_files(outdir: Path) -> None:
    # This function is intentionally restricted to the dedicated local
    # threshold-sweep folder.
    outdir = outdir.resolve()
    if SENS_RUNS_DIR.resolve() not in outdir.parents:
        raise RuntimeError(f"Refusing to clean a non-sweep directory: {outdir}")
    patterns = (
        "*_perframe.csv",
        "*_kinetics.csv",
        "*_spots_debug.tif",
        "aggregated_*.csv",
    )
    for pattern in patterns:
        for path in outdir.glob(pattern):
            path.unlink()


def run_one_threshold(
    sweep_type: str,
    threshold: float,
    gene: str,
    root_dir: str,
) -> dict:
    global THRESH_MCP, THRESH_TETR, TETR_RESCUE_THR
    global USE_TETR_RESCUE, SAVE_DEBUG_IMAGES

    config = THRESHOLD_SWEEPS[sweep_type]
    outdir = _threshold_run_dir(sweep_type, gene, threshold)
    outdir.mkdir(parents=True, exist_ok=True)
    required = [
        outdir / "aggregated_perframe.csv",
        outdir / "aggregated_runs_by_cell.csv",
    ]
    if all(path.exists() for path in required) and not OVERWRITE_EXISTING_SWEEPS:
        print(f"[reuse] {sweep_type} {threshold:.3f} {gene}: {outdir}")
        return {
            "sweep_type": sweep_type,
            "threshold": threshold,
            "gene": gene,
            "status": "reused",
            "output_dir": str(outdir),
            "n_input_files": np.nan,
        }

    if OVERWRITE_EXISTING_SWEEPS:
        _remove_generated_threshold_files(outdir)

    saved_globals = (
        THRESH_MCP,
        THRESH_TETR,
        TETR_RESCUE_THR,
        USE_TETR_RESCUE,
        SAVE_DEBUG_IMAGES,
    )
    try:
        if sweep_type == "mtetr":
            THRESH_TETR = float(threshold)
            THRESH_MCP = float(config["fixed_mcp"])
            USE_TETR_RESCUE = False
        elif sweep_type == "mcp":
            THRESH_MCP = float(threshold)
            THRESH_TETR = float(config["fixed_tetr"])
            USE_TETR_RESCUE = bool(config["use_tetr_rescue"])
        else:
            raise ValueError(f"Unknown sweep type: {sweep_type}")

        # Rescue threshold must track the mTetR threshold used in this run.
        TETR_RESCUE_THR = THRESH_TETR * 0.85
        SAVE_DEBUG_IMAGES = False

        root = Path(root_dir)
        files = sorted(
            path for path in root.glob("*.tif")
            if not path.name.endswith("_labels.tif")
        )
        if not files:
            raise FileNotFoundError(f"No input TIFF files found in {root}")

        errors = []
        print(
            f"[run] {sweep_type}={threshold:.3f} | {gene} | "
            f"{len(files)} files -> {outdir}"
        )
        for path in files:
            try:
                process_tiff_file(path, outdir)
            except Exception as exc:
                errors.append({"file": str(path), "error": repr(exc)})

        if errors:
            pd.DataFrame(errors).pipe(export_transcription_states).to_csv(
                outdir / "processing_errors.csv", index=False
            )
            raise RuntimeError(
                f"{len(errors)} input file(s) failed for "
                f"{sweep_type}={threshold:.3f}, {gene}."
            )

        _aggregate_threshold_perframe(outdir)
        build_runs_by_cell(outdir, DELTA_MIN)
        return {
            "sweep_type": sweep_type,
            "threshold": threshold,
            "gene": gene,
            "status": "completed",
            "output_dir": str(outdir),
            "n_input_files": len(files),
        }
    finally:
        (
            THRESH_MCP,
            THRESH_TETR,
            TETR_RESCUE_THR,
            USE_TETR_RESCUE,
            SAVE_DEBUG_IMAGES,
        ) = saved_globals


sweep_manifest_rows = []
for sweep_type, config in THRESHOLD_SWEEPS.items():
    for threshold in config["values"]:
        for gene, root_dir in DATASETS:
            sweep_manifest_rows.append(
                run_one_threshold(sweep_type, threshold, gene, root_dir)
            )

sweep_manifest = pd.DataFrame(sweep_manifest_rows)
sweep_manifest.pipe(export_transcription_states).to_csv(
    SENS_TABLES_DIR / "threshold_sweep_manifest.csv", index=False
)
display(sweep_manifest)


In [ ]:
# Collect cell-level Active fractions and censoring-aware run data.
def _boolean_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes"})
    )


def collect_threshold_sweeps() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    active_parts = []
    run_parts = []
    qc_rows = []

    for sweep_type, config in THRESHOLD_SWEEPS.items():
        for threshold in config["values"]:
            for gene, _ in DATASETS:
                outdir = _threshold_run_dir(sweep_type, gene, threshold)
                perframe_path = outdir / "aggregated_perframe.csv"
                runs_path = outdir / "aggregated_runs_by_cell.csv"
                if not perframe_path.exists() or not runs_path.exists():
                    raise FileNotFoundError(
                        f"Missing threshold output in {outdir}; run the sweep cell first."
                    )

                perframe = _ensure_cell_uid(read_legacy_state_csv(perframe_path))
                perframe["group"] = perframe["group"].replace(
                    {"RGFP966_0h": "RGFP966"}
                )
                perframe = perframe[
                    perframe["group"].isin(SENS_GROUPS)
                    & np.isfinite(perframe["on_off"])
                ].copy()
                cell_values = (
                    perframe.groupby(["group", "cell_uid"], as_index=False)
                    .agg(
                        n_frames=("on_off", "size"),
                        active_fraction=("on_off", "mean"),
                    )
                )
                cell_values = cell_values[
                    cell_values["n_frames"] >= MIN_CELL_FRAMES
                ].copy()
                cell_values["sweep_type"] = sweep_type
                cell_values["threshold"] = float(threshold)
                cell_values["gene"] = gene
                active_parts.append(cell_values)

                runs = _ensure_cell_uid(read_legacy_state_csv(runs_path))
                runs["group"] = runs["group"].replace(
                    {"RGFP966_0h": "RGFP966"}
                )
                runs["left_censored"] = _boolean_series(
                    runs["left_censored"]
                )
                runs["right_censored"] = _boolean_series(
                    runs["right_censored"]
                )
                runs = runs[
                    runs["group"].isin(SENS_GROUPS)
                    & np.isfinite(runs["length_min"])
                    & (runs["length_min"] > 0)
                ].copy()
                runs["sweep_type"] = sweep_type
                runs["threshold"] = float(threshold)
                runs["gene"] = gene
                run_parts.append(runs)

                for group in SENS_GROUPS:
                    group_cells = cell_values[cell_values["group"] == group]
                    group_runs = runs[runs["group"] == group]
                    qc_rows.append(
                        {
                            "sweep_type": sweep_type,
                            "threshold": float(threshold),
                            "gene": gene,
                            "group": group,
                            "n_cells": group_cells["cell_uid"].nunique(),
                            "n_active_runs": int((group_runs["kind"] == "ON").sum()),
                            "n_inactive_runs": int((group_runs["kind"] == "OFF").sum()),
                            "n_left_censored": int(group_runs["left_censored"].sum()),
                            "n_right_censored": int(group_runs["right_censored"].sum()),
                        }
                    )

    active = pd.concat(active_parts, ignore_index=True)
    runs = pd.concat(run_parts, ignore_index=True)
    qc = pd.DataFrame(qc_rows)
    return active, runs, qc


active_by_cell, runs_by_cell, threshold_qc = collect_threshold_sweeps()
active_by_cell.pipe(export_transcription_states).to_csv(
    SENS_TABLES_DIR / "threshold_sensitivity_active_fraction_by_cell.csv",
    index=False,
)
runs_by_cell.pipe(export_transcription_states).to_csv(
    SENS_TABLES_DIR / "threshold_sensitivity_runs_by_cell.csv",
    index=False,
)
threshold_qc.pipe(export_transcription_states).to_csv(
    SENS_TABLES_DIR / "threshold_sensitivity_qc.csv", index=False
)
display(threshold_qc)


In [ ]:
# Estimate drug-minus-DMSO effects with cell-level bootstrap confidence intervals.
def _stable_seed(*parts: object) -> int:
    token = "|".join(map(str, parts)).encode("utf-8")
    digest = hashlib.sha256(token).hexdigest()
    return (BOOTSTRAP_SEED + int(digest[:8], 16)) % (2**32 - 1)


def _bootstrap_p_two_sided(samples: np.ndarray) -> float:
    samples = samples[np.isfinite(samples)]
    if samples.size == 0:
        return np.nan
    lower = (np.count_nonzero(samples <= 0) + 1) / (samples.size + 1)
    upper = (np.count_nonzero(samples >= 0) + 1) / (samples.size + 1)
    return min(1.0, 2.0 * min(lower, upper))


def _holm_adjust(pvalues: pd.Series) -> np.ndarray:
    values = pvalues.to_numpy(dtype=float)
    adjusted = np.full(values.shape, np.nan, dtype=float)
    valid = np.flatnonzero(np.isfinite(values))
    if valid.size == 0:
        return adjusted
    order = valid[np.argsort(values[valid])]
    running = 0.0
    m = len(order)
    for rank, index in enumerate(order):
        candidate = min(1.0, values[index] * (m - rank))
        running = max(running, candidate)
        adjusted[index] = running
    return adjusted


def _mean_difference_bootstrap(
    control: np.ndarray,
    treatment: np.ndarray,
    rng: np.random.Generator,
) -> tuple[float, float, float, float]:
    control = np.asarray(control, dtype=float)
    treatment = np.asarray(treatment, dtype=float)
    estimate = float(np.mean(treatment) - np.mean(control))
    samples = np.empty(BOOTSTRAP_ITERATIONS, dtype=float)
    for index in range(BOOTSTRAP_ITERATIONS):
        control_sample = rng.choice(control, size=len(control), replace=True)
        treatment_sample = rng.choice(
            treatment, size=len(treatment), replace=True
        )
        samples[index] = (
            np.mean(treatment_sample) - np.mean(control_sample)
        )
    low, high = np.percentile(samples, [2.5, 97.5])
    return estimate, float(low), float(high), _bootstrap_p_two_sided(samples)


def _rmst_km(
    durations: np.ndarray,
    events: np.ndarray,
    tau: float,
) -> float:
    durations = np.asarray(durations, dtype=float)
    events = np.asarray(events, dtype=bool)
    valid = np.isfinite(durations) & (durations > 0)
    durations = durations[valid]
    events = events[valid]
    if durations.size == 0:
        return np.nan

    survival = 1.0
    previous_time = 0.0
    area = 0.0
    for time in np.unique(durations[durations <= tau]):
        area += survival * (float(time) - previous_time)
        at_risk = int(np.count_nonzero(durations >= time))
        observed = int(np.count_nonzero((durations == time) & events))
        if at_risk > 0:
            survival *= 1.0 - observed / at_risk
        previous_time = float(time)
    area += survival * max(0.0, tau - previous_time)
    return float(area)


def _run_clusters(runs: pd.DataFrame) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    clusters = {}
    for cell_uid, cell_runs in runs.groupby("cell_uid", sort=False):
        clusters[str(cell_uid)] = (
            cell_runs["length_min"].to_numpy(dtype=float),
            (~cell_runs["right_censored"]).to_numpy(dtype=bool),
        )
    return clusters


def _rmst_from_clusters(
    clusters: dict[str, tuple[np.ndarray, np.ndarray]],
    sampled_ids=None,
) -> float:
    ids = np.asarray(list(clusters) if sampled_ids is None else sampled_ids)
    durations = np.concatenate([clusters[str(cell)][0] for cell in ids])
    events = np.concatenate([clusters[str(cell)][1] for cell in ids])
    return _rmst_km(durations, events, RMST_TAU_MIN)


def _rmst_difference_cluster_bootstrap(
    control_runs: pd.DataFrame,
    treatment_runs: pd.DataFrame,
    rng: np.random.Generator,
) -> tuple[float, float, float, float, int, int]:
    control_clusters = _run_clusters(control_runs)
    treatment_clusters = _run_clusters(treatment_runs)
    control_ids = np.asarray(list(control_clusters))
    treatment_ids = np.asarray(list(treatment_clusters))
    if control_ids.size == 0 or treatment_ids.size == 0:
        return np.nan, np.nan, np.nan, np.nan, 0, 0

    estimate = (
        _rmst_from_clusters(treatment_clusters)
        - _rmst_from_clusters(control_clusters)
    )
    samples = np.empty(BOOTSTRAP_ITERATIONS, dtype=float)
    for index in range(BOOTSTRAP_ITERATIONS):
        sampled_control = rng.choice(
            control_ids, size=control_ids.size, replace=True
        )
        sampled_treatment = rng.choice(
            treatment_ids, size=treatment_ids.size, replace=True
        )
        samples[index] = (
            _rmst_from_clusters(treatment_clusters, sampled_treatment)
            - _rmst_from_clusters(control_clusters, sampled_control)
        )
    low, high = np.percentile(samples, [2.5, 97.5])
    return (
        float(estimate),
        float(low),
        float(high),
        _bootstrap_p_two_sided(samples),
        int(control_ids.size),
        int(treatment_ids.size),
    )


def calculate_threshold_effects(
    active: pd.DataFrame,
    runs: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    for sweep_type, config in THRESHOLD_SWEEPS.items():
        for threshold in config["values"]:
            for gene, _ in DATASETS:
                active_subset = active[
                    (active["sweep_type"] == sweep_type)
                    & np.isclose(active["threshold"], threshold)
                    & (active["gene"] == gene)
                ]
                control_active = active_subset[
                    active_subset["group"] == "DMSO"
                ]["active_fraction"].to_numpy(dtype=float)
                for comparison in SENS_COMPARISONS:
                    treatment_active = active_subset[
                        active_subset["group"] == comparison
                    ]["active_fraction"].to_numpy(dtype=float)
                    rng = np.random.default_rng(
                        _stable_seed(
                            sweep_type,
                            threshold,
                            gene,
                            comparison,
                            "active_fraction",
                        )
                    )
                    estimate, low, high, pvalue = _mean_difference_bootstrap(
                        control_active, treatment_active, rng
                    )
                    rows.append(
                        {
                            "sweep_type": sweep_type,
                            "threshold": threshold,
                            "gene": gene,
                            "comparison": f"{comparison} - DMSO",
                            "metric": "active_fraction",
                            "estimate": estimate,
                            "ci_low": low,
                            "ci_high": high,
                            "p_bootstrap": pvalue,
                            "n_control_cells": len(control_active),
                            "n_treatment_cells": len(treatment_active),
                            "rmst_tau_min": np.nan,
                        }
                    )

                    for state, metric in (
                        ("ON", "active_rmst"),
                        ("OFF", "inactive_rmst"),
                    ):
                        run_subset = runs[
                            (runs["sweep_type"] == sweep_type)
                            & np.isclose(runs["threshold"], threshold)
                            & (runs["gene"] == gene)
                            & (runs["kind"] == state)
                            & (~runs["left_censored"])
                        ]
                        control_runs = run_subset[
                            run_subset["group"] == "DMSO"
                        ]
                        treatment_runs = run_subset[
                            run_subset["group"] == comparison
                        ]
                        rng = np.random.default_rng(
                            _stable_seed(
                                sweep_type,
                                threshold,
                                gene,
                                comparison,
                                metric,
                            )
                        )
                        (
                            estimate,
                            low,
                            high,
                            pvalue,
                            n_control,
                            n_treatment,
                        ) = _rmst_difference_cluster_bootstrap(
                            control_runs, treatment_runs, rng
                        )
                        rows.append(
                            {
                                "sweep_type": sweep_type,
                                "threshold": threshold,
                                "gene": gene,
                                "comparison": f"{comparison} - DMSO",
                                "metric": metric,
                                "estimate": estimate,
                                "ci_low": low,
                                "ci_high": high,
                                "p_bootstrap": pvalue,
                                "n_control_cells": n_control,
                                "n_treatment_cells": n_treatment,
                                "rmst_tau_min": RMST_TAU_MIN,
                            }
                        )

    effects = pd.DataFrame(rows)
    effects["q_holm"] = np.nan
    correction_groups = ["sweep_type", "gene", "metric"]
    for _, indices in effects.groupby(correction_groups).groups.items():
        effects.loc[indices, "q_holm"] = _holm_adjust(
            effects.loc[indices, "p_bootstrap"]
        )
    return effects


threshold_effects = calculate_threshold_effects(
    active_by_cell, runs_by_cell
)
threshold_effects.pipe(export_transcription_states).to_csv(
    SENS_TABLES_DIR / "threshold_sensitivity_effects.csv", index=False
)
display(threshold_effects)


In [ ]:
# One compact supplementary figure: mTetR block (a) and MCP block (b).
def plot_threshold_sensitivity_summary(effects: pd.DataFrame) -> plt.Figure:
    colors = {
        "TSA - DMSO": "#E69F00",
        "RGFP966 - DMSO": "#009E73",
    }
    metric_specs = [
        ("active_fraction", "Delta Active fraction"),
        ("active_rmst", f"Delta Active RMST (min; tau={RMST_TAU_MIN:g})"),
        ("inactive_rmst", f"Delta Inactive RMST (min; tau={RMST_TAU_MIN:g})"),
    ]
    row_specs = [
        ("mtetr", "Nanog"),
        ("mtetr", "Sox2"),
        ("mcp", "Nanog"),
        ("mcp", "Sox2"),
    ]

    plt.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )
    figure, axes = plt.subplots(
        len(row_specs),
        len(metric_specs),
        figsize=(7.2, 7.6),
        sharex=False,
        constrained_layout=True,
    )

    for row_index, (sweep_type, gene) in enumerate(row_specs):
        config = THRESHOLD_SWEEPS[sweep_type]
        values = np.asarray(config["values"], dtype=float)
        for column_index, (metric, ylabel) in enumerate(metric_specs):
            axis = axes[row_index, column_index]
            axis.axhline(0, color="black", linewidth=0.8, linestyle="--")
            axis.axvline(
                config["paper_value"],
                color="#D62728",
                linewidth=1.0,
                linestyle="-",
                label="Main-analysis threshold",
            )
            axis.axvline(
                config["snapshot_value"],
                color="#7F7F7F",
                linewidth=1.0,
                linestyle=":",
                label="Snapshot threshold",
            )

            for comparison in SENS_COMPARISONS:
                comparison_label = f"{comparison} - DMSO"
                subset = effects[
                    (effects["sweep_type"] == sweep_type)
                    & (effects["gene"] == gene)
                    & (effects["metric"] == metric)
                    & (effects["comparison"] == comparison_label)
                ].sort_values("threshold")
                x = subset["threshold"].to_numpy(dtype=float)
                y = subset["estimate"].to_numpy(dtype=float)
                low = subset["ci_low"].to_numpy(dtype=float)
                high = subset["ci_high"].to_numpy(dtype=float)
                axis.plot(
                    x,
                    y,
                    marker="o",
                    markersize=3.5,
                    linewidth=1.25,
                    color=colors[comparison_label],
                    label=comparison_label,
                )
                axis.fill_between(
                    x,
                    low,
                    high,
                    color=colors[comparison_label],
                    alpha=0.18,
                    linewidth=0,
                )

            axis.set_xticks(values)
            axis.set_xticklabels([f"{value:g}" for value in values])
            axis.grid(axis="y", linestyle=":", alpha=0.45)
            axis.spines["top"].set_visible(False)
            axis.spines["right"].set_visible(False)
            if row_index in (1, 3):
                threshold_name = "mTetR" if sweep_type == "mtetr" else "MCP"
                axis.set_xlabel(f"{threshold_name} normalized threshold")
            if column_index == 0:
                threshold_name = "mTetR" if sweep_type == "mtetr" else "MCP"
                axis.set_ylabel(f"{gene}\n{ylabel}")
            else:
                axis.set_ylabel(ylabel)
            if row_index == 0:
                axis.set_title(metric_specs[column_index][1])

    axes[0, 0].text(
        -0.34,
        1.22,
        "a",
        transform=axes[0, 0].transAxes,
        fontsize=12,
        fontweight="bold",
    )
    axes[2, 0].text(
        -0.34,
        1.22,
        "b",
        transform=axes[2, 0].transAxes,
        fontsize=12,
        fontweight="bold",
    )
    figure.text(0.51, 1.005, "mTetR threshold sensitivity", ha="center", fontsize=10)
    figure.text(0.51, 0.505, "MCP threshold sensitivity", ha="center", fontsize=10)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    figure.legend(
        unique.values(),
        unique.keys(),
        loc="upper center",
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, 1.045),
    )
    return figure


threshold_figure = plot_threshold_sensitivity_summary(threshold_effects)
pdf_path = SENS_FIGS_DIR / "R3_m11_threshold_sensitivity_summary.pdf"
png_path = SENS_FIGS_DIR / "R3_m11_threshold_sensitivity_summary.png"
threshold_figure.savefig(pdf_path, bbox_inches="tight")
threshold_figure.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved PDF: {pdf_path.resolve()}")
print(f"Saved PNG: {png_path.resolve()}")
print(f"Source tables: {SENS_TABLES_DIR.resolve()}")


In [ ]:
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np


def plot_threshold_sensitivity_summary_revised(
    effects,
    panel_label_fontsize=11,
    font_size=8,
):
    colors = {
        "TSA - DMSO": "#E69F00",
        "RGFP966 - DMSO": "#009E73",
    }
    metric_specs = [
        ("active_fraction", "Delta Active fraction"),
        ("active_rmst", f"Delta Active RMST (min; tau={RMST_TAU_MIN:g})"),
        ("inactive_rmst", f"Delta Inactive RMST (min; tau={RMST_TAU_MIN:g})"),
    ]
    row_specs = [
        ("mtetr", "Nanog"),
        ("mtetr", "Sox2"),
        ("mcp", "Nanog"),
        ("mcp", "Sox2"),
    ]

    plt.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
            "font.size": font_size,
            "axes.titlesize": font_size + 0.5,
            "axes.labelsize": font_size,
            "xtick.labelsize": font_size - 0.5,
            "ytick.labelsize": font_size - 0.5,
            "legend.fontsize": font_size - 0.5,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )

    figure = plt.figure(figsize=(7.4, 8.2))
    grid = figure.add_gridspec(
        nrows=5,
        ncols=3,
        height_ratios=[1.0, 1.0, 0.18, 1.0, 1.0],
        left=0.11,
        right=0.98,
        bottom=0.13,
        top=0.89,
        hspace=0.46,
        wspace=0.32,
    )

    grid_rows = [0, 1, 3, 4]
    axes = np.empty((4, 3), dtype=object)
    for column_index in range(3):
        axes[0, column_index] = figure.add_subplot(grid[0, column_index])
        axes[1, column_index] = figure.add_subplot(
            grid[1, column_index],
            sharex=axes[0, column_index],
            sharey=axes[0, column_index],
        )
        axes[2, column_index] = figure.add_subplot(
            grid[3, column_index],
            sharey=axes[0, column_index],
        )
        axes[3, column_index] = figure.add_subplot(
            grid[4, column_index],
            sharex=axes[2, column_index],
            sharey=axes[0, column_index],
        )

    for row_index, (sweep_type, gene) in enumerate(row_specs):
        config = THRESHOLD_SWEEPS[sweep_type]
        threshold_values = np.asarray(config["values"], dtype=float)
        threshold_name = "mTetR" if sweep_type == "mtetr" else "MCP"

        for column_index, (metric, column_title) in enumerate(metric_specs):
            axis = axes[row_index, column_index]
            axis.axhline(0, color="black", linewidth=0.8, linestyle="--", zorder=1)
            axis.axvline(
                config["paper_value"],
                color="#D62728",
                linewidth=1.0,
                zorder=2,
            )
            axis.axvline(
                config["snapshot_value"],
                color="#7F7F7F",
                linewidth=1.0,
                linestyle=":",
                zorder=2,
            )

            for comparison in SENS_COMPARISONS:
                comparison_label = f"{comparison} - DMSO"
                subset = effects[
                    (effects["sweep_type"] == sweep_type)
                    & (effects["gene"] == gene)
                    & (effects["metric"] == metric)
                    & (effects["comparison"] == comparison_label)
                ].sort_values("threshold")

                if subset.empty:
                    continue

                x = subset["threshold"].to_numpy(dtype=float)
                estimate = subset["estimate"].to_numpy(dtype=float)
                ci_low = subset["ci_low"].to_numpy(dtype=float)
                ci_high = subset["ci_high"].to_numpy(dtype=float)

                axis.fill_between(
                    x,
                    ci_low,
                    ci_high,
                    color=colors[comparison_label],
                    alpha=0.18,
                    linewidth=0,
                    zorder=3,
                )
                axis.plot(
                    x,
                    estimate,
                    color=colors[comparison_label],
                    marker="o",
                    markersize=3.5,
                    linewidth=1.25,
                    zorder=4,
                )

            axis.set_xticks(threshold_values)
            axis.set_xticklabels([f"{value:g}" for value in threshold_values])
            axis.grid(axis="y", linestyle=":", linewidth=0.7, alpha=0.45)
            axis.spines["top"].set_visible(False)
            axis.spines["right"].set_visible(False)
            axis.margins(x=0.05)

            if row_index in (0, 2):
                axis.set_title(column_title, pad=5)
                axis.tick_params(axis="x", labelbottom=False)
            if row_index in (1, 3):
                axis.set_xlabel(f"{threshold_name} normalized threshold")

            if column_index == 0:
                axis.set_ylabel(rf"$\mathit{{{gene}}}$" "\nDrug - DMSO", labelpad=6)

    # Panel labels and block titles are placed in reserved whitespace.
    top_block_y = axes[0, 0].get_position().y1 + 0.045
    lower_block_y = axes[2, 0].get_position().y1 + 0.045
    figure.text(0.02, top_block_y, "a", fontsize=panel_label_fontsize, fontweight="bold")
    figure.text(
        0.5,
        top_block_y,
        "mTetR threshold sensitivity",
        ha="center",
        fontsize=font_size + 0.5,
    )
    figure.text(0.02, lower_block_y, "b", fontsize=panel_label_fontsize, fontweight="bold")
    figure.text(
        0.5,
        lower_block_y,
        "MCP threshold sensitivity",
        ha="center",
        fontsize=font_size + 0.5,
    )

    legend_handles = [
        Line2D([0], [0], color="#D62728", linewidth=1.0),
        Line2D([0], [0], color="#7F7F7F", linewidth=1.0, linestyle=":"),
        Line2D([0], [0], color=colors["TSA - DMSO"], marker="o", linewidth=1.25),
        Line2D(
            [0],
            [0],
            color=colors["RGFP966 - DMSO"],
            marker="o",
            linewidth=1.25,
        ),
    ]
    legend_labels = [
        "Main-analysis threshold",
        "Snapshot threshold",
        "TSA - DMSO",
        "RGFP966 - DMSO",
    ]
    figure.legend(
        legend_handles,
        legend_labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.035),
        ncol=4,
        frameon=False,
        handlelength=2.6,
        columnspacing=1.5,
    )
    return figure


revised_figure = plot_threshold_sensitivity_summary_revised(threshold_effects)
revised_pdf_path = (
    SENS_FIGS_DIR / "R3_m11_threshold_sensitivity_summary_revised.pdf"
)
revised_png_path = (
    SENS_FIGS_DIR / "R3_m11_threshold_sensitivity_summary_revised.png"
)
revised_figure.savefig(revised_pdf_path, bbox_inches="tight")
revised_figure.savefig(revised_png_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved PDF: {revised_pdf_path.resolve()}")
print(f"Saved PNG: {revised_png_path.resolve()}")
